



































 # Amazon ML Challenge 2026 — Business Entity Resolution
**Pipeline (built for ~2M S1 × ~10M S2/S3 records):** Load from Google Drive → Normalize (incl. Tamil/Hindi
transliteration) → Blocking: 2 approximate-nearest-neighbour blockers (FAISS) + rare-name-token + address-key
→ Vectorized pairwise features → LightGBM (GroupKFold by S1) → thresholds tuned on macro F0.5 → one-to-one assignment.
Test is processed in chunks of S1 so it fits in Colab RAM.

**v7:** train now has the same share of *orphan* S2/S3 records (copies of businesses with no S1 record) as test. Test has ~23% more S2/S3 records per S1 than train in every country, and the leaderboard sat ~1 point under the out-of-fold score. Section 4b hides some S1 records outside the training sample so their copies become orphans; section 12 estimates the orphan share of test without labels, and section 13 compares predicted matches per S1 on test with the out-of-fold ones. Bigger cross-encoder training pool on big machines.

**v6:** two new blockers, a **reverse kNN** (every S2/S3 record searches all S1 of its country, so copies crowded out of their S1's own top-K are found from their side) and a **learned retrieval encoder** (section 5b, a small neural network trained contrastively on true pairs), plus features from both on every pair (`dl_cos`, `dl_t_gap`, `dl_nn_margin`, `rev_rank`), a finer threshold search, and the blocking recall check now prints the F0.5 ceiling.

**v5:** learned map for state names written in Indian scripts (23% of India S2/S3 addresses), honorific prefixes (M/s, Smt, Shri, Dr ...) stripped from names, OCR digit fixes (5hree → shree).

Runs on **Google Colab** (data downloaded from your Drive, results copied back to Drive) or locally; it detects which one by itself and sizes the run to the machine's RAM.

Run top to bottom. No external lookup APIs are used. Libraries: LightGBM (MIT), FAISS (MIT), RapidFuzz (MIT), anyascii (ISC).

## 0. Install packages (Colab already has pandas / sklearn / lightgbm; this adds the rest)

In [ ]:
%pip install -q rapidfuzz lightgbm pyarrow faiss-cpu anyascii psutil xgboost transformers

In [ ]:
%pip install scikit-learn lightgbm

## 1. Imports & configuration

In [ ]:
import os, sys, re, gc, csv, time, json, shutil, zipfile, subprocess, warnings
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import GroupKFold
import lightgbm as lgb
from rapidfuzz import fuzz
from rapidfuzz.distance import JaroWinkler, Levenshtein
warnings.filterwarnings("ignore")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "true")   # HF tokenizers: fast batches, no warning on fork

IN_COLAB = "google.colab" in sys.modules   # detected: Colab downloads the data from Drive (section 2)
SEED = 42
np.random.seed(SEED)

# ---- LOCAL runs only: where your data is, the zip OR the extracted folder (Colab uses DRIVE_LINK in section 2) ----
DATA_PATH = r"C:\Users\kusha\Downloads\student_resource.zip"
# DATA_PATH = r"C:\Users\kusha\Downloads\student_resource\student_resource"   # already-extracted folder

# ---- Blocking config (tune with the recall report in section 8) ----
BLOCK_BY_COUNTRY = True   # EDA: country agreement inside matched pairs = 1.0, so this is safe
K_NAME  = 20              # kNN on business-name vectors, per source (20 / 30 = the 0.9908-ceiling Colab run)
K_FULL  = 30              # kNN on name + address vectors, per source
K_RARE  = 8               # shared rare name token (incl. name without spaces, catches website names)
K_AKEY  = 8               # address key: house number + street word, or PIN / postal code
K_ADDR  = 8               # kNN on the address alone: same place, changed / made-up name, no house number needed
K_NAKEY = 8               # rare 'name word | place word' keys (quartz|crittenden): common name, same locality
ADDR_MIN = 0.5            # address-kNN neighbours below this cosine are dropped (empty / generic addresses)
SVD_DIM, PQ_M, NPROBE = 96, 48, 96   # vector size, PQ compression, IVF cells searched (was 12 = only ~0.35% of each index)
IVF_MIN = 50000           # groups smaller than this use exact search
RARE_MAX_DF = 50          # a token/key counts as "rare" if it appears in at most this many target records
USE_PHON, K_PHON = True, 8                # phonetic name keys (laxmi / lakshmi, bhavani / bawani)
LEARN_TRANSLIT = True                     # section 5: learn 'tredrs → traders' from matched train pairs
LEARN_STATES   = True                     # section 5: learn 'महाराष्ट्र → mh' (state names in Indian scripts)
USE_HOP, K_HOP, HOP_SEED_MIN = True, 8, 0.6   # second hop: neighbours of each S1's best match (other copies)

# ---- v6: reverse kNN + learned retrieval encoder (sections 5b and 6) ----
USE_REV, K_REV = True, 3              # reverse kNN: every S2/S3 record keeps its K_REV closest S1 among ALL S1 of its country
REV_FIELDS = ("full", "dl")           # vectors the reverse search uses: name + address (SVD) and the learned encoder
USE_DL, K_DL = True, 20               # learned encoder (5b): its own kNN blocker + learned-similarity features on every pair
DL_DIM, DL_BAG, DL_BUCKETS = 96, 128, 2**20                      # output vector, token-embedding size, hashed token buckets
DL_POOL_S1, DL_EPOCHS, DL_BATCH, DL_LR = 600_000, 2, 4096, 2e-3  # trained on true pairs of S1 OUTSIDE the GBDT sample

# ---- Blocking recall check (section 6b): measures recall on 30k train S1 and can pick K / hop for you ----
RUN_RECALL_CHECK = True
RECALL_CHECK_N   = 30000
AUTO_BLOCKING    = True       # True = section 6b sets K_NAME, K_FULL, USE_HOP to the best option it measured
BIG_K            = (20, 30)   # (K_NAME, K_FULL) tried as the "bigger K" option
MAX_CANDS_PER_S1 = 110        # never auto-pick an option with more candidates than this (test time grows with it)
MIN_RECALL_GAIN  = 0.004      # a more expensive option must add at least this much recall to be picked

# ---- Sizes / modelling (lower TRAIN_S1_FRAC and S1_CHUNK if your PC has 8 GB RAM or less) ----
MATCH_TEST_CROWD = True   # v7 (4b): hide some train S1 outside the sample → train gets test's share of orphan S2/S3
TRAIN_S1_FRAC = 0.08      # sample of TRAIN S1 entities (≈175k; was 0.05). Lower to 0.05 on MemoryError. ALL S2/S3 records are kept (realistic negatives).
S1_CHUNK = 40000          # S1 records processed per chunk
N_FOLDS = 3
KEEP_P = 0.02             # test pairs with p below this are dropped before the final decision (saves RAM)
UNSEEN_COUNTRY_MARGIN = 0.03
RUN_LOCO = False          # already measured: india 0.845, us 0.925
USE_S1_NN = True          # each S1's closest look-alike S1 (full S1 table) → 'is the copy closer to me or to it?'
RUN_ERROR_ANALYSIS = True # section 10c: where the F0.5 is lost (train out-of-fold), before memory is freed
TRY_STAGE2 = True         # section 10b: stage-2 model + per-country thresholds (used only if OOF improves)
USE_CE = True             # section 10a: fine-tuned cross-encoder re-ranker → stage 2 (GPU runtimes only)
CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"   # 22M parameters, Apache-2.0
CE_POOL_S1 = 300_000      # S1 outside the GBDT training sample whose candidate pairs train the cross-encoder
CE_NEG_PER_S1, CE_RAND_NEG_PER_S1 = 12, 4   # hardest negatives (blocker similarity) + random negatives per S1
CE_EPOCHS, CE_LR, CE_BATCH, CE_MAXLEN = 1, 5e-5, 256, 96

def f05_entity(pred, true):
    """Exact per-entity F0.5 as defined in the problem statement."""
    pred, true = set(pred), set(true)
    if not true and not pred: return 1.0
    if not true or not pred: return 0.0
    tp = len(pred & true)
    if tp == 0: return 0.0
    p, r = tp / len(pred), tp / len(true)
    return 1.25 * p * r / (0.25 * p + r)

def macro_f05(pred_map, gt):
    return float(np.mean([f05_entity(pred_map.get(k, []), v) for k, v in gt.items()]))

def tic(msg):
    """Tiny timer/logger."""
    print(f"[{time.strftime('%H:%M:%S')}] {msg}", flush=True)

try:
    import psutil
    RAM_GB = psutil.virtual_memory().total / 1e9
    print(f"CPU cores: {os.cpu_count()} | RAM: {RAM_GB:.1f} GB total, "
          f"{psutil.virtual_memory().available/1e9:.1f} GB free")
except ImportError:
    RAM_GB = 0
    print(f"CPU cores: {os.cpu_count()} (pip install psutil to also see RAM)")

# ---- Bigger machine → bigger training sample, and allow the "bigger K" blocking option (+0.7 pts recall in v5) ----
if RAM_GB >= 120:      # e.g. Colab Pro G4 / A100 runtime (~175 GB)
    TRAIN_S1_FRAC, MAX_CANDS_PER_S1, S1_CHUNK = 0.25, 160, 80000
    DL_POOL_S1 = 1_500_000
    CE_POOL_S1 = 600_000   # big GPU: twice the cross-encoder training pool (its loss was still falling at 300k)
elif RAM_GB >= 45:     # e.g. Colab CPU runtime with High-RAM
    TRAIN_S1_FRAC, MAX_CANDS_PER_S1 = 0.10, 150
    DL_POOL_S1 = 1_000_000
print(f"{'Colab' if IN_COLAB else 'local'} run | TRAIN_S1_FRAC={TRAIN_S1_FRAC}  MAX_CANDS_PER_S1={MAX_CANDS_PER_S1}  "
      f"S1_CHUNK={S1_CHUNK}")

# ---- All CPU cores for the text steps (normalization, n-gram hashing): the GPU cannot do those ----
import multiprocessing as mp
N_PROC = min(os.cpu_count() or 1, 32) if "fork" in mp.get_all_start_methods() else 1   # Windows: 1 process
_PAR = {}
def _par_job(ab):
    a, b = ab; d = _PAR["data"]
    return _PAR["func"](d.iloc[a:b] if hasattr(d, "iloc") else d[a:b])
def par_map(func, data, min_chunk=100_000, timeout=1800):
    """func(data[a:b]) on up to N_PROC forked processes, results joined in order (DataFrame / sparse / array).
    Workers inherit func and data through fork (nothing big is pickled on the way in).
    Any failure or a hang longer than `timeout` seconds → runs in this process instead."""
    n = len(data); k = min(N_PROC, n // min_chunk)
    if k < 2: return func(data)
    b = np.linspace(0, n, k + 1).astype(int)
    _PAR.update(func=func, data=data)
    try:
        with mp.get_context("fork").Pool(k) as pool:
            parts = pool.map_async(_par_job, list(zip(b[:-1], b[1:]))).get(timeout)
    except Exception as e:
        print(f"  (parallel step failed, running in one process: {repr(e)[:150]})")
        return func(data)
    finally:
        _PAR.clear()
    if isinstance(parts[0], pd.DataFrame): return pd.concat(parts, ignore_index=True)
    if hasattr(parts[0], "tocsr"):
        from scipy import sparse
        return sparse.vstack(parts, format="csr")
    return np.concatenate(parts)
print(f"text steps use {N_PROC} CPU process{'es' if N_PROC > 1 else ''}")
try:   # many cores + several thread pools (BLAS, PyTorch, FAISS, XGBoost) → cap BLAS so they don't fight
    from threadpoolctl import threadpool_limits
    threadpool_limits(limits=min(16, os.cpu_count() or 1), user_api="blas")
except Exception:
    pass

# ---- GPU (Colab G4 / A100 / L4 / T4): exact nearest-neighbour search (PyTorch) + XGBoost training/prediction ----
USE_GPU = True            # False = CPU only: FAISS IVF-PQ + LightGBM, exactly as in v5
USE_XGB = True            # False = keep LightGBM (CPU) as the model even on a GPU runtime (kNN still uses the GPU)
KNN_DEVICE, GPU_KNN, MODEL = "cuda", False, "lgb"
KNN_SCORE_ELEMS = 2e9     # kNN scores held per GPU batch (float16: 2e9 = 4 GB); 1e9 on GPUs under 40 GB
if USE_GPU:
    try:
        import torch
        if torch.cuda.is_available():
            _a = torch.randn(64, 96, device="cuda", dtype=torch.float16)
            torch.topk(_a @ _a.T, 5, dim=1); torch.cuda.synchronize()          # fails early on an unsupported GPU
            GPU_KNN = True
            if torch.cuda.get_device_properties(0).total_memory < 40e9: KNN_SCORE_ELEMS = 1e9
            print(f"GPU: {torch.cuda.get_device_name(0)} | "
                  f"{torch.cuda.get_device_properties(0).total_memory/1e9:.0f} GB → exact kNN blocking on GPU")
    except Exception as e:
        print("GPU kNN not available, using FAISS on CPU:", repr(e)[:200])
    try:
        import xgboost as xgb
        if GPU_KNN and USE_XGB:
            _X = np.random.rand(2000, 5).astype(np.float32); _y = (_X[:, 0] > 0.5).astype(np.int8)
            xgb.train(dict(device="cuda", tree_method="hist", objective="binary:logistic", verbosity=0),
                      xgb.DMatrix(_X, _y), 2)
            MODEL = "xgb"
            print(f"XGBoost {xgb.__version__} → model training + prediction on GPU")
    except Exception as e:
        print("XGBoost GPU not available, using LightGBM on CPU:", repr(e)[:200])
def gpu_mem():
    """Device-wide GPU memory in use (PyTorch + XGBoost + CuPy), for the progress lines."""
    if not GPU_KNN: return "no GPU"
    free, total = torch.cuda.mem_get_info()
    return f"GPU {(total - free) / 1e9:.1f}/{total / 1e9:.0f} GB"
print(f"blocking kNN: {'GPU exact (PyTorch)' if GPU_KNN else 'FAISS IVF-PQ (CPU)'} | model: "
      f"{'XGBoost (GPU)' if MODEL == 'xgb' else 'LightGBM (CPU)'}")
if USE_CE:   # the cross-encoder needs a GPU and the transformers library
    try:
        import transformers
        USE_CE = bool(GPU_KNN and TRY_STAGE2)
    except ImportError:
        USE_CE = False
    if USE_CE: KEEP_P = 0.005   # keep more low-score pairs, so the cross-encoder can rescue them
    if USE_CE: MIN_RECALL_GAIN = 0.002   # with re-ranking, extra recall is worth more candidates
print(f"cross-encoder re-ranker (10a): {'ON → ' + CE_MODEL if USE_CE else 'off'} | KEEP_P={KEEP_P}")

## 2. Get the data
**Colab:** paste the link of `student_resource.zip` into `DRIVE_LINK`. The zip is downloaded **by its file ID** with your
own Google login (a popup asks once), so the file does not need to be public and a file shared with you works too.
It is copied to Colab's fast local disk (~1 GB), extracted (~2.5 GB), and Drive stays mounted to save the results.
Re-running skips anything already done; after the runtime is deleted the local disk is wiped and it downloads again (1-3 min).

**Local:** uses `DATA_PATH` from section 1.

In [ ]:
# ---- Colab: your student_resource.zip in Google Drive (share link or just the file ID) ----
DRIVE_LINK     = ""   # Optional: paste your student_resource.zip Drive link or file ID here
DRIVE_ZIP_PATH = ""   # optional instead of the link: e.g. "/content/drive/MyDrive/student_resource.zip"

def extract_zip(zpath, dest):
    """Unzip once into `dest`, skipping macOS junk (__MACOSX/, .DS_Store). A marker file prevents re-extracting."""
    marker = os.path.join(dest, ".extracted_" + os.path.basename(zpath))
    if os.path.exists(marker):
        print("already extracted →", dest); return
    need = sum(i.file_size for i in zipfile.ZipFile(zpath).infolist())
    free = shutil.disk_usage(dest).free
    print(f"extracting {need/1e9:.2f} GB (disk free: {free/1e9:.1f} GB) → {dest}")
    if free < need * 1.2: raise OSError("Not enough disk space to extract the zip")
    tic("extracting ...")
    with zipfile.ZipFile(zpath) as z:
        for m in z.namelist():
            if "__MACOSX" in m or m.endswith(".DS_Store"): continue
            z.extract(m, dest)
            if m.endswith(".tsv"): print("   ", m)
    open(marker, "w").close()
    tic("extracted")

def find_base(root):
    """Locate the folder that contains dataset/train/train_source1.tsv (the student_resource dir)."""
    for dp, dns, fns in os.walk(root):
        dns[:] = [d for d in dns if d not in ("__MACOSX", "cache", "runs", "output", "drive")]   # don't wander
        if "train_source1.tsv" in fns and os.path.basename(dp) == "train":
            return os.path.dirname(os.path.dirname(os.path.abspath(dp)))
    raise FileNotFoundError(f"dataset/train/train_source1.tsv not found under {root}")

def copy_with_progress(src, dst, chunk=64 * 1024 * 1024):
    """Copy a big file in 64 MB chunks, printing progress."""
    total = os.path.getsize(src); done = 0
    with open(src, "rb") as fi, open(dst + ".part", "wb") as fo:
        while True:
            b = fi.read(chunk)
            if not b: break
            fo.write(b); done += len(b)
            print(f"\r  copied {done/1e9:.2f} / {total/1e9:.2f} GB", end="")
    os.replace(dst + ".part", dst); print()

def download_drive_file(file_id, dst, chunk=100 * 1024 * 1024):
    """Download a (private or shared-with-you) file from Drive by file ID using the Drive API, in 100 MB chunks."""
    from google.colab import auth
    from googleapiclient.discovery import build
    from googleapiclient.http import MediaIoBaseDownload
    auth.authenticate_user()                      # a popup asks you to allow access, once per session
    svc = build("drive", "v3")
    meta = svc.files().get(fileId=file_id, fields="name,size", supportsAllDrives=True).execute()
    print(f"Downloading '{meta['name']}' ({int(meta.get('size', 0))/1e9:.2f} GB) ...")
    req = svc.files().get_media(fileId=file_id, supportsAllDrives=True)
    with open(dst + ".part", "wb") as fh:
        dl, done = MediaIoBaseDownload(fh, req, chunksize=chunk), False
        while not done:
            status, done = dl.next_chunk()
            if status: print(f"\r  {status.progress()*100:5.1f}%", end="")
    os.replace(dst + ".part", dst); print("\n  done")

def drive_file_id(link):
    """Accepts .../file/d/<ID>/view, ...?id=<ID>, or the bare ID."""
    m = re.search(r"/d/([\w-]{20,})|[?&]id=([\w-]{20,})", link)
    return (m.group(1) or m.group(2)) if m else link.strip()

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")                 # also used to save the results (sections 12 and 14)
    LOCAL_ROOT = "/content/data"                  # Colab's fast local disk
    os.makedirs(LOCAL_ROOT, exist_ok=True)
    ZIP_LOCAL = os.path.join(LOCAL_ROOT, "student_resource.zip")
    if not (os.path.exists(ZIP_LOCAL) and zipfile.is_zipfile(ZIP_LOCAL)):
        if DRIVE_ZIP_PATH and os.path.exists(DRIVE_ZIP_PATH):
            copy_with_progress(DRIVE_ZIP_PATH, ZIP_LOCAL)
        else:
            download_drive_file(drive_file_id(DRIVE_LINK), ZIP_LOCAL)
    assert zipfile.is_zipfile(ZIP_LOCAL), "Downloaded file is not a zip - check DRIVE_LINK"
    extract_zip(ZIP_LOCAL, LOCAL_ROOT)
else:
    if not os.path.exists(DATA_PATH):
        raise FileNotFoundError(f"DATA_PATH does not exist: {DATA_PATH}\n"
                                "Right-click the zip in File Explorer → 'Copy as path', and paste it into DATA_PATH.")
    if DATA_PATH.lower().endswith(".zip"):
        LOCAL_ROOT = os.path.join(os.path.dirname(os.path.abspath(DATA_PATH)), "er_data")
        os.makedirs(LOCAL_ROOT, exist_ok=True)
        extract_zip(DATA_PATH, LOCAL_ROOT)
    else:
        LOCAL_ROOT = os.path.abspath(DATA_PATH)

BASE = find_base(LOCAL_ROOT)
TRAIN_DIR, TEST_DIR = os.path.join(BASE, "dataset", "train"), os.path.join(BASE, "dataset", "test")
OUT_DIR, CACHE_DIR = os.path.join(BASE, "output"), os.path.join(BASE, "cache")
os.makedirs(OUT_DIR, exist_ok=True); os.makedirs(CACHE_DIR, exist_ok=True)
print("BASE  =", BASE)
print("train:", sorted(os.listdir(TRAIN_DIR)))
print("test :", sorted(os.listdir(TEST_DIR)))
print(f"disk free: {shutil.disk_usage(BASE).free/1e9:.1f} GB (keep ~8 GB free for cache + candidate file)")


## 3. Load TSVs (fast + memory-safe, with a parquet cache)
* `dtype=str, keep_default_na=False` → IDs and PIN codes stay strings, empty stays `""` (not NaN).
* `quoting=QUOTE_NONE` → a stray `"` in a name cannot swallow the following lines.
* First load is cached to parquet; re-runs load in seconds.

In [ ]:
def count_lines(path):
    """Count data lines (minus header) quickly, to verify the TSV parsed correctly."""
    n = 0
    with open(path, "rb") as f:
        for b in iter(lambda: f.read(1 << 24), b""): n += b.count(b"\n")
    return n - 1

def read_tsv(path):
    """Read a challenge TSV as strings (IDs/PINs keep leading zeros), cache it as parquet for fast re-runs."""
    cache = os.path.join(CACHE_DIR, os.path.basename(path) + ".parquet")
    if os.path.exists(cache): return pd.read_parquet(cache)
    df = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE, encoding="utf-8")
    n_lines = count_lines(path)
    if abs(len(df) - n_lines) > 1: print(f"  WARNING {path}: {len(df)} rows vs {n_lines} lines - check quoting")
    df = df.fillna("")
    try: df = df.astype("string[pyarrow]")   # ~3x less RAM than Python object strings
    except Exception: pass
    try: df.to_parquet(cache, index=False)
    except Exception as e: print("  (parquet cache skipped:", e, ")")
    return df

tic("loading TRAIN (first run ~2-4 min, later runs use the parquet cache)")
tr1, tr2, tr3 = [read_tsv(os.path.join(TRAIN_DIR, f"train_source{i}.tsv")) for i in (1, 2, 3)]
gt_raw = read_tsv(os.path.join(TRAIN_DIR, "train_ground_truth.tsv"))
# Test files are loaded later (section 12) after train memory is freed.
for n, d in [("tr1", tr1), ("tr2", tr2), ("tr3", tr3), ("gt", gt_raw)]:
    print(f"{n}: {len(d):>10,} rows  mem={d.memory_usage(deep=True).sum()/1e6:,.0f} MB")
for i in (1, 2, 3):
    print(f"te{i}: {count_lines(os.path.join(TEST_DIR, f'test_source{i}.tsv')):>10,} rows (counted, not loaded)")
try:
    import psutil; vm = psutil.virtual_memory()
    print(f"RAM: {vm.available/1e9:.1f} GB free of {vm.total/1e9:.1f} GB")
except ImportError: pass
tr1.head()

## 4. EDA — answers that decide the design

In [ ]:
gt = {r.source1_entity_id: [x for x in str(r.matched_entity_ids).split(",") if x] for r in gt_raw.itertuples()}
n_match = pd.Series({k: len(v) for k, v in gt.items()})
print("S1 entities:", len(gt), "| singleton %:", round((n_match == 0).mean() * 100, 2))
print("matches per S1:\n", n_match.value_counts().sort_index().head(10))
all_m = pd.Series([x for v in gt.values() for x in v])
print("S2 share of matches:", round(all_m.str.startswith("S2-").mean(), 3))
dup = all_m.duplicated().sum()
print("S2/S3 IDs matched to >1 S1:", dup, "→ one-to-one assignment is", "SAFE" if dup == 0 else "NOT exact")

cmap = pd.concat([tr1, tr2, tr3]).set_index("entity_id")["country"]
pairs_c = [(cmap.get(k), cmap.get(x)) for k, v in gt.items() for x in v]
print("country agreement inside matched pairs:", round(np.mean([a == b for a, b in pairs_c]), 4))
te_countries = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t", usecols=["country"], dtype=str,
                           keep_default_na=False, quoting=csv.QUOTE_NONE).country.value_counts()
print("train countries:", tr1.country.value_counts().to_dict(), "| test S1 countries:", te_countries.to_dict())

nm = pd.concat([tr2, tr3]).set_index("entity_id")
for k in list(k for k, v in gt.items() if v)[:5]:
    print("\nS1:", tr1.loc[tr1.entity_id == k, ["business_name", "business_address"]].values[0])
    for x in gt[k]: print("  ↳", x, nm.loc[x, ["business_name", "business_address"]].values)

### 4b. Train sample: a fraction of S1, but ALL of S2/S3
Keeping every S2/S3 record means each sampled S1 sees exactly the crowd of similar-but-wrong records it would see
in the full data (the true matches of other S1 entities become its negatives, just like in test).

**v7, `MATCH_TEST_CROWD`:** test has ~23% more S2/S3 records per S1 than train, in every country (5.75 vs 4.68), which
means more *orphans*: copies of businesses that have no S1 record, whose closest S1 is a look-alike. Train gets the
same ratio by hiding S1 records that are **outside** the training sample (they are never queries, so no label changes):
their copies stay in S2/S3 as orphans, and the hidden S1 are left out of everything that models the S1 crowd (look-alikes,
reverse kNN, name counts, encoder and cross-encoder pools). Expect a lower out-of-fold score that is closer to the
leaderboard.


In [ ]:
rng = np.random.default_rng(SEED)
keep1 = rng.random(len(tr1)) < TRAIN_S1_FRAC
drop1 = np.zeros(len(tr1), bool)                 # v7: S1 hidden from the train S1 universe (never in the sample)
if MATCH_TEST_CROWD:
    _cc = lambda i: pd.read_csv(os.path.join(TEST_DIR, f"test_source{i}.tsv"), sep="\t", usecols=["country"], dtype=str,
                                keep_default_na=False, quoting=csv.QUOTE_NONE).country
    te_t = pd.concat([_cc(2), _cc(3)], ignore_index=True).value_counts()
    tr_t = pd.concat([tr2.country.astype(str), tr3.country.astype(str)], ignore_index=True).value_counts()
    c1 = tr1.country.astype(str).to_numpy(dtype=object)
    for c in sorted(set(c1)):
        n1 = int((c1 == c).sum())
        if c not in te_countries or c not in te_t or c not in tr_t: continue
        r_tr, r_te = tr_t[c] / n1, te_t[c] / te_countries[c]         # S2/S3 records per S1
        cand = np.flatnonzero((c1 == c) & ~keep1)
        n_drop = int(np.clip(round(n1 - tr_t[c] / r_te), 0, len(cand)))
        drop1[rng.choice(cand, n_drop, replace=False)] = True
        print(f"  {c}: S2/S3 per S1 train {r_tr:.2f} vs test {r_te:.2f} → hiding {n_drop:,} of {n1:,} train S1 (outside "
              f"the sample) → train {tr_t[c] / (n1 - n_drop):.2f}")
    del te_t, tr_t, c1
# S1 entities NOT in the training sample are used only to learn the transliteration dictionary (section 5).
# Keeping them apart means the out-of-fold score stays honest.
tr1_dict = tr1.loc[~keep1, ["entity_id", "business_name", "business_address"]].reset_index(drop=True)
_hidden = set(tr1.entity_id.to_numpy(dtype=object)[drop1].tolist())
tr1_all = tr1[~drop1]                           # train S1 universe: look-alikes, reverse kNN, name counts, pools
T_OWNED_IDS = np.array([x for k, v in gt.items() if k not in _hidden for x in v], dtype=object)   # S2/S3 with an S1
tr1 = tr1[keep1].reset_index(drop=True)
keep_ids = set(tr1.entity_id.tolist())
gt_dict = {k: v for k, v in gt.items() if v and k not in keep_ids}
gt = {k: v for k, v in gt.items() if k in keep_ids}
del _hidden
for _v in ["gt_raw", "cmap", "nm", "pairs_c", "all_m", "n_match"]: globals().pop(_v, None)
gc.collect()
print(f"train S1 sample: {len(tr1):,}  | S2: {len(tr2):,}  S3: {len(tr3):,}  | true pairs: {sum(map(len, gt.values())):,}"
      f"  | S1 kept for the dictionary: {len(tr1_dict):,}  | S1 hidden (orphan copies): {int(drop1.sum()):,}")


## 5. Normalization
Transliterates Indian scripts to Latin with anyascii, then fixes its biggest weakness: anyascii drops vowels
("लक्ष्मी ट्रेडर्स" → "lksmi tredrs"). A dictionary **learned from matched training pairs** maps those tokens back to
the usual English spelling ("tredrs" → "traders"). It is learned only from S1 entities outside the training sample.
Also strips accents (French), removes "null", website parts, ordinals ("45nd" → "45"), leading zeros ("0684" → "684"),
expands abbreviations and maps state names to codes. State names written in an Indian script ("महाराष्ट्र", "ಕರ್ನಾಟಕ") are mapped to the state code with a second
learned map. Honorific prefixes added by the noisy sources ("M/s", "Smt", "Shri", "Dr") are dropped and OCR digit
swaps inside words are fixed ("5hree" → "shree", "de1hi" → "delhi"). Adds a phonetic-skeleton name (`name_ph`) and an Indic-script flag.


In [ ]:
from anyascii import anyascii
from rapidfuzz import fuzz
from rapidfuzz.process import cpdist

INDIC_RE = "[\u0900-\u0DFF]"   # Devanagari, Bengali, Gurmukhi, Gujarati, Odia, Tamil, Telugu, Kannada, Malayalam
TMAP = globals().get("TMAP", {})   # learned transliteration dictionary (filled below)

# phonetic skeleton: merge transliteration variants, drop vowels, collapse double letters
PHON_RULES = [("ksh", "ks"), ("x", "ks"), ("sh", "s"), ("ch", "c"), ("ck", "k"), ("ph", "f"), ("th", "t"),
              ("dh", "d"), ("bh", "b"), ("kh", "k"), ("gh", "g"), ("jh", "j"), ("w", "v"), ("z", "j"), ("q", "k")]
_ANUSVARA = re.compile(r"m(?=[tdkgcjs])")     # anyascii writes the nasal as m: "emtrpraijej" → "entrpraijej"
_VOWELS, _DOUBLE = re.compile(r"[aeiouy]"), re.compile(r"(.)\1+")
def phon(w):
    for a, b in PHON_RULES: w = w.replace(a, b)
    w = _ANUSVARA.sub("n", w)
    w = w[0] + _VOWELS.sub("", w[1:])
    return _DOUBLE.sub(r"\1", w)
def phon_str(s):
    return " ".join(phon(w) for w in s.split())

NAME_ABBR = {"corp": "corporation", "co": "company", "pvt": "private", "ltd": "limited", "intl": "international",
             "mfg": "manufacturing", "svcs": "services", "svc": "service", "bros": "brothers", "assoc": "associates",
             "mgmt": "management", "ent": "enterprises", "ind": "industries", "natl": "national", "cie": "compagnie",
             "praivet": "private", "elelpi": "llp", "limitid": "limited", "pvtltd": "private limited"}
ADDR_ABBR = {"rd": "road", "st": "street", "str": "street", "ave": "avenue", "av": "avenue", "blvd": "boulevard",
             "bd": "boulevard", "ln": "lane", "dr": "drive", "ct": "court", "hwy": "highway", "pkwy": "parkway",
             "sq": "square", "apt": "apartment", "ste": "suite", "fl": "floor", "flr": "floor", "bldg": "building",
             "opp": "opposite", "nr": "near", "sec": "sector", "ph": "phase", "dist": "district", "n": "north",
             "s": "south", "e": "east", "w": "west", "ne": "northeast", "nw": "northwest", "se": "southeast",
             "sw": "southwest", "pl": "place", "ch": "chemin", "imp": "impasse", "rte": "route", "fg": "faubourg",
             "mt": "mount", "ctr": "center", "centre": "center", "twp": "township", "terr": "terrace", "ter": "terrace",
             "cir": "circle", "pk": "park", "sqr": "square"}
US_STATES = {"alabama": "al", "alaska": "ak", "arizona": "az", "arkansas": "ar", "california": "ca", "colorado": "co",
    "connecticut": "ct", "delaware": "de", "district of columbia": "dc", "florida": "fl", "georgia": "ga", "hawaii": "hi",
    "idaho": "id", "illinois": "il", "indiana": "in", "iowa": "ia", "kansas": "ks", "kentucky": "ky", "louisiana": "la",
    "maine": "me", "maryland": "md", "massachusetts": "ma", "michigan": "mi", "minnesota": "mn", "mississippi": "ms",
    "missouri": "mo", "montana": "mt", "nebraska": "ne", "nevada": "nv", "new hampshire": "nh", "new jersey": "nj",
    "new mexico": "nm", "new york": "ny", "north carolina": "nc", "north dakota": "nd", "ohio": "oh", "oklahoma": "ok",
    "oregon": "or", "pennsylvania": "pa", "rhode island": "ri", "south carolina": "sc", "south dakota": "sd",
    "tennessee": "tn", "texas": "tx", "utah": "ut", "vermont": "vt", "virginia": "va", "washington": "wa",
    "west virginia": "wv", "wisconsin": "wi", "wyoming": "wy"}
IN_STATES = {"andhra pradesh": "ap", "arunachal pradesh": "ar", "assam": "as", "bihar": "br", "chhattisgarh": "cg",
    "goa": "ga", "gujarat": "gj", "haryana": "hr", "himachal pradesh": "hp", "jharkhand": "jh", "karnataka": "ka",
    "kerala": "kl", "madhya pradesh": "mp", "maharashtra": "mh", "manipur": "mn", "meghalaya": "ml", "mizoram": "mz",
    "nagaland": "nl", "odisha": "od", "orissa": "od", "punjab": "pb", "rajasthan": "rj", "sikkim": "sk",
    "tamil nadu": "tn", "tamilnadu": "tn", "tmilnatu": "tn", "telangana": "tg", "ts": "tg", "tripura": "tr",
    "uttar pradesh": "up", "uttr prdes": "up", "uttarakhand": "uk", "uttaranchal": "uk", "west bengal": "wb",
    "jammu and kashmir": "jk", "ladakh": "la", "puducherry": "py", "pondicherry": "py", "chandigarh": "ch",
    "delhi": "dl"}
# US and Indian codes overlap (e.g. "tn"); harmless because blocking and matching stay inside one country.
STATE_MAP = {**US_STATES, **IN_STATES}
LEGAL = {"inc", "incorporated", "llc", "llp", "ltd", "limited", "private", "pvt", "corporation", "corp", "company",
         "co", "plc", "lp", "pc", "pllc", "opc", "sarl", "sas", "sasu", "sa", "eurl", "sci", "snc", "scop", "gmbh",
         "ag", "bv", "nv", "pte", "pty", "srl", "spa", "societe", "compagnie", "the", "and", "of", "et", "de", "la",
         "le", "les", "du", "des", "d", "l"}
DBA_RE = re.compile(r"\b(?:dba|d b a|doing business as|trading as|aka|a k a|fka|f k a|formerly known as|formerly)\b")
WEB_RE = re.compile(r"(?:www\.|https?://|\.(?:com|net|org|biz|info|in|fr|co)\b)", re.I)

def _abbr_re(d): return re.compile(r"\b(" + "|".join(map(re.escape, sorted(d, key=len, reverse=True))) + r")\b")
NAME_ABBR_RE, ADDR_ABBR_RE, STATE_RE = _abbr_re(NAME_ABBR), _abbr_re(ADDR_ABBR), _abbr_re(STATE_MAP)
# prefixes the noisy sources add in front of names (each on ~1.6% of India copies); stripped on both sides
HONOR_RE = re.compile(r"^(?:(?:m s|mr|mrs|ms|smt|dr|sri|shri)\s+)+(?=\S)")
OCR_RE, OCR_FIX = re.compile(r"\b[a-z015]*[015][a-z015]*\b"), str.maketrans("015", "ols")
def _ocr(m):   # "5hree" → "shree", "de1hi" → "delhi"; numbers and short codes stay as they are
    w = m.group(0)
    return w.translate(OCR_FIX) if sum(c.isalpha() for c in w) >= 3 else w

def base_norm(s: pd.Series) -> pd.Series:
    """Transliterate non-Latin scripts + accents to ASCII (only rows that need it), lowercase, clean punctuation/nulls."""
    s = s.fillna("").astype(str).astype(object)          # object dtype → Python `re` on every pandas version
    na = s.str.contains(r"[^\x00-\x7f]", regex=True)
    if na.any(): s[na] = s[na].map(anyascii)
    s = s.str.lower().str.replace("&", " and ", regex=False).str.replace(r"['`´]", "", regex=True)
    s = s.str.replace(r"[^a-z0-9]+", " ", regex=True)
    s = s.str.replace(r"\b(?:null|none|nan|n a)\b", " ", regex=True)
    return s.str.replace(r"\s+", " ", regex=True).str.strip()

def _prep_chunk(df):
    out = pd.DataFrame({"eid": df["entity_id"].astype(str).values})
    out["country_key"] = base_norm(df["country"]).values
    raw = df["business_name"].fillna("").astype(str).astype(object)
    name = base_norm(raw)
    ind = raw.str.contains(INDIC_RE, regex=True)
    if TMAP and ind.any():   # "lksmi tredrs" → "lakshmi traders"
        name[ind] = name[ind].map(lambda s: " ".join(TMAP.get(w, w) for w in s.split()))
    web = raw.str.contains(WEB_RE)
    if web.any():  # "maurewilliamscolombier.com" → "maurewilliamscolombier"
        name[web] = name[web].str.replace(r"^(?:https?\s)?www\s|\s(?:com|net|org|biz|info|in|fr|co)(?:\s(?:in|uk))?$",
                                          "", regex=True).str.strip()
    name = name.str.replace(OCR_RE, _ocr, regex=True).str.replace(HONOR_RE, "", regex=True)
    name = name.str.replace(NAME_ABBR_RE, lambda m: NAME_ABBR[m.group(1)], regex=True)
    out["name_n"] = name.values
    alt = name.str.split(DBA_RE, n=1, regex=True).str[-1].str.strip()
    out["name_alt"] = np.where(alt.values != name.values, alt.values, "")
    core = name.str.split().apply(lambda t: " ".join(w for w in t if w not in LEGAL) or " ".join(t))
    out["name_core"] = core.values
    out["acronym"] = core.str.split().apply(lambda t: "".join(w[0] for w in t) if len(t) >= 2 else "").values
    out["is_web"] = web.astype(np.int8).values
    out["is_indic"] = ind.astype(np.int8).values
    out["name_ph"] = core.map(phon_str).values
    addr = base_norm(df["business_address"])
    addr = addr.str.replace(r"\b(\d+)(?:st|nd|rd|th)\b", r"\1", regex=True)      # 45nd / 45th → 45
    addr = addr.str.replace(r"\b0+(\d)", r"\1", regex=True)                       # 0684 → 684
    addr = addr.str.replace(r"\b(\d{3}) (\d{3})\b", r"\1\2", regex=True)          # "110 001" → "110001"
    addr = addr.str.replace(ADDR_ABBR_RE, lambda m: ADDR_ABBR[m.group(1)], regex=True)
    addr = addr.str.replace(STATE_RE, lambda m: STATE_MAP[m.group(1)], regex=True)
    out["addr_n"] = addr.values
    nums = addr.str.findall(r"\b\d+\b")
    out["postal"] = nums.apply(lambda l: next((x for x in reversed(l) if 5 <= len(x) <= 6), "")).values
    out["first_num"] = nums.apply(lambda l: l[0] if l else "").values
    out["nums"] = nums.str.join(" ").values                                        # "1291 1 1"
    return out

def _prep_typed(df):
    """_prep_chunk + compact column types (also runs inside the worker processes)."""
    p = _prep_chunk(df)
    for c in p.columns:
        if c == "country_key": p[c] = p[c].astype("category")
        elif c not in ("is_web", "is_indic"):
            try: p[c] = p[c].astype("string[pyarrow]")
            except Exception: pass
    return p

def prep_records(df, chunk=500_000):
    """Normalize a whole source table (split over all CPU cores on Colab/Linux); compact Arrow-string columns."""
    if N_PROC > 1:
        out = par_map(_prep_typed, df, min_chunk=50_000)
    else:
        parts = []
        for a in range(0, len(df), chunk):
            parts.append(_prep_typed(df.iloc[a:a + chunk]))
            if len(df) > chunk: print(f"\r   {min(a + chunk, len(df)):,}/{len(df):,}", end="")
        if len(df) > chunk: print()
        out = pd.concat(parts, ignore_index=True)
    out["country_key"] = out["country_key"].astype(str).astype("category")
    return out

def prep_stage(s1, s2, s3):
    """Q = normalized S1; T = normalized S2 + S3 in one table (row number = global target id)."""
    tic("  S1"); Q = prep_records(s1)
    tic("  S2"); T2 = prep_records(s2)
    tic("  S3"); T3 = prep_records(s3)
    T2["src"] = np.int8(2); T3["src"] = np.int8(3)
    T = pd.concat([T2, T3], ignore_index=True); T["country_key"] = T["country_key"].astype(str).astype("category")
    del T2, T3; gc.collect()
    return Q, T

def learn_translit(tr1_d, gt_d, tr2, tr3, min_ent=3, min_purity=0.5):
    """Learn 'anyascii token → English spelling' from true pairs (Latin S1 name ↔ Indic-script S2/S3 name).
    Tokens are aligned by position when both names have the same number of words; a mapping is kept if at least
    `min_ent` different S1 entities support it and it wins at least `min_purity` of that token's alignments."""
    T = pd.concat([tr2[["entity_id", "business_name"]], tr3[["entity_id", "business_name"]]], ignore_index=True)
    ind = T.business_name.astype(str).str.contains(INDIC_RE, regex=True).values
    print(f"  Indic-script names in train S2+S3: {ind.mean()*100:.1f}%  ({ind.sum():,} records)")
    tname = pd.Series(T.business_name.values[ind], index=T.entity_id.astype(str).values[ind])
    del T; gc.collect()
    nat = set(tname.index)
    P = pd.DataFrame([(k, x) for k, v in gt_d.items() for x in v if x in nat], columns=["s1", "t"])
    if len(P) == 0: return {}, tname
    s1name = pd.Series(tr1_d.business_name.values, index=tr1_d.entity_id.astype(str).values)
    a = base_norm(pd.Series(s1name.reindex(P.s1.values).values)).str.split().values
    b = base_norm(pd.Series(tname.reindex(P.t.values).values)).str.split().values
    rows = [(w2, w1, s) for s, x, y in zip(P.s1.values, a, b) if len(x) == len(y)
            for w1, w2 in zip(x, y) if w1.isalpha() and w2.isalpha()]
    print(f"  aligned {len(rows):,} word pairs from {len(P):,} Indic-script true pairs")
    R = pd.DataFrame(rows, columns=["tok", "eng", "s1"]).drop_duplicates()
    del rows, a, b, P; gc.collect()
    g = R.groupby(["tok", "eng"]).size().rename("n").reset_index()      # n = distinct S1 entities
    g["purity"] = g.n / g.groupby("tok").n.transform("sum")
    g = g.sort_values("n", ascending=False).drop_duplicates("tok")
    g = g[(g.n >= min_ent) & (g.purity >= min_purity) & (g.tok != g.eng)]
    print(f"  learned {len(g):,} mappings, e.g.:",
          ", ".join(f"{t}→{e}" for t, e in zip(g.tok.head(25), g.eng.head(25))))
    return dict(zip(g.tok, g.eng)), tname

INDIC_RUN = re.compile(INDIC_RE + "[\u0900-\u0DFF\\s\u200c\u200d]*")
IN_CODES = set(IN_STATES.values())

def learn_states(tr1_d, gt_d, tr2, tr3, max_pairs=300_000, min_n=20, min_purity=0.6):
    """Learn 'state name in an Indian script → state code' from true pairs: the S1 address has the state in English.
    Only a handful of distinct strings exist (one per state and script), so a few thousand pairs are plenty."""
    T = pd.concat([tr2[["entity_id", "business_address"]], tr3[["entity_id", "business_address"]]], ignore_index=True)
    ind = T.business_address.astype(str).str.contains(INDIC_RE, regex=True).values
    print(f"  Indic-script text in train S2+S3 addresses: {ind.mean()*100:.1f}%  ({ind.sum():,} records)")
    taddr = pd.Series(T.business_address.values[ind], index=T.entity_id.astype(str).values[ind])
    del T; gc.collect()
    nat = set(taddr.index)
    P = pd.DataFrame([(k, x) for k, v in gt_d.items() for x in v if x in nat], columns=["s1", "t"])
    if len(P) == 0: return {}
    P = P.sample(min(max_pairs, len(P)), random_state=SEED)
    s1addr = pd.Series(tr1_d.business_address.values, index=tr1_d.entity_id.astype(str).values)
    a = base_norm(pd.Series(s1addr.reindex(P.s1.values).values))
    a = a.str.replace(STATE_RE, lambda m: STATE_MAP[m.group(1)], regex=True).str.split().values
    runs = [INDIC_RUN.findall(str(s)) for s in taddr.reindex(P.t.values).values]
    uniq = sorted({r for rr in runs for r in rr})
    key = dict(zip(uniq, base_norm(pd.Series(uniq))))          # normalize each distinct string once
    rows = [(key[r], c) for x, rr in zip(a, runs) for c in set(x) & IN_CODES for r in rr if key[r]]
    g = pd.DataFrame(rows, columns=["key", "code"]).groupby(["key", "code"]).size().rename("n").reset_index()
    g["purity"] = g.n / g.groupby("key").n.transform("sum")
    g = g.sort_values("n", ascending=False).drop_duplicates("key")
    g = g[(g.n >= min_n) & (g.purity >= min_purity) & (g.key != g.code)]
    M = {k: c for k, c in zip(g.key, g.code) if STATE_MAP.get(k) != c}
    print(f"  learned {len(M)} state names:", ", ".join(f"{k}→{c}" for k, c in M.items()))
    return M

def check_translit(M, tr1_s, gt_s, tname):
    """Name similarity of the TRAINING SAMPLE's Indic-script true pairs, before vs after the dictionary."""
    nat = set(tname.index)
    P = pd.DataFrame([(k, x) for k, v in gt_s.items() for x in v if x in nat], columns=["s1", "t"])
    if len(P) == 0: print("  no Indic-script pairs in the training sample"); return
    s1name = pd.Series(tr1_s.business_name.values, index=tr1_s.entity_id.astype(str).values)
    a = base_norm(pd.Series(s1name.reindex(P.s1.values).values)).tolist()
    b = base_norm(pd.Series(tname.reindex(P.t.values).values))
    b2 = b.map(lambda s: " ".join(M.get(w, w) for w in s.split())).tolist()
    s0 = cpdist(a, b.tolist(), scorer=fuzz.token_sort_ratio, workers=-1).mean()
    s1 = cpdist(a, b2, scorer=fuzz.token_sort_ratio, workers=-1).mean()
    n_true = sum(map(len, gt_s.values()))
    print(f"  training sample: {len(P):,} of {n_true:,} true pairs ({len(P)/max(n_true,1)*100:.1f}%) have Indic-script "
          f"names | name similarity {s0:.1f} → {s1:.1f} /100 with the dictionary")

if LEARN_TRANSLIT:
    tic("learning transliteration dictionary")
    TMAP, _tname = learn_translit(tr1_dict, gt_dict, tr2, tr3)
    check_translit(TMAP, tr1, gt, _tname)
    del _tname
if globals().get("LEARN_STATES", True):
    tic("learning state names written in Indian scripts")
    STATE_MAP.update(learn_states(tr1_dict, gt_dict, tr2, tr3))
    STATE_RE = _abbr_re(STATE_MAP)   # _prep_chunk reads STATE_RE / STATE_MAP at call time, so train and test both use it
for _v in (["tr1_dict"] if (USE_CE or USE_DL) else ["tr1_dict", "gt_dict"]): globals().pop(_v, None)   # 5b / 10a need gt_dict
gc.collect()

tic("normalizing TRAIN"); Q_tr, T_tr = prep_stage(tr1, tr2, tr3)
if USE_S1_NN or USE_CE or USE_DL or USE_REV:
    tic("  full train S1 (look-alikes, reverse kNN, encoder / cross-encoder pools)"); S1ALL_tr = prep_records(tr1_all)
del tr1_all
del tr2, tr3; gc.collect()
TRAIN_COUNTRIES = set(Q_tr.country_key.astype(str))
print(f"Q_tr {len(Q_tr):,}  T_tr {len(T_tr):,}  mem {(Q_tr.memory_usage(deep=True).sum() + T_tr.memory_usage(deep=True).sum())/1e9:.2f} GB")
Q_tr[["eid", "name_n", "name_core", "addr_n", "postal", "first_num"]].head(8)

## 5b. Learned retrieval encoder (deep learning)
A small neural encoder, trained from scratch on the true pairs of ~1M S1 records **outside** the GBDT training sample
(so its scores stay honest features, like the cross-encoder in 10a). A record = hashed char 2-4-grams + words of the
name, and char 3-grams + words of the address → two embedding bags → MLP → 96-d unit vector; S1 and S2/S3 share it.
Loss: InfoNCE in both directions with in-batch negatives. Batches are grouped by the S1 name's first word or by its place
word, so the other records in a batch are look-alikes, the same crowd blocking has to beat.

Used in section 6 as a kNN blocker (`ann_dl`), inside the reverse kNN, and for three features on every candidate pair:
`dl_cos` (learned similarity), `dl_t_gap` (how far this S1 is behind the copy's closest S1 among ALL S1) and
`dl_nn_margin` (closer to this S1 than to its look-alike S1?). A few minutes on a GPU, ~20 min on a CPU.


In [ ]:
# ---- 5b. Learned retrieval encoder ----
from sklearn.feature_extraction.text import HashingVectorizer
DL_ENC = None

def _grams(s, ns):
    """Char n-grams of every word (with word boundaries) + the whole word."""
    out = []
    for w in s.split():
        x = " " + w + " "
        for n in ns:
            out.extend(x[i:i + n] for i in range(len(x) - n + 1))
        out.append("W" + w)
    return out

def dl_name_an(s):
    """Name tokens: char 2-4-grams, whole words, and the name without spaces (website-style names)."""
    out = _grams(s, (2, 3, 4))
    if " " in s: out.append("N" + s.replace(" ", ""))
    return out

def dl_addr_an(s):
    """Address tokens: char 3-grams + whole words (house numbers, PIN codes, street and place names)."""
    return _grams(s, (3,))

_HV_N = HashingVectorizer(analyzer=dl_name_an, n_features=DL_BUCKETS, binary=True, norm=None,
                          alternate_sign=False, dtype=np.float32)
_HV_A = HashingVectorizer(analyzer=dl_addr_an, n_features=DL_BUCKETS, binary=True, norm=None,
                          alternate_sign=False, dtype=np.float32)

def dl_hash(names, addrs):
    """Hashed token bags (CSR) of names and addresses, on all CPU cores."""
    return (par_map(_HV_N.transform, list(names), min_chunk=50_000).tocsr(),
            par_map(_HV_A.transform, list(addrs), min_chunk=50_000).tocsr())

if USE_DL:
    try:
        import torch
        import torch.nn.functional as TF
        DL_DEVICE = KNN_DEVICE if GPU_KNN else "cpu"
        if DL_DEVICE == "cpu":
            DL_EPOCHS = min(DL_EPOCHS, 1); print("(no GPU: the encoder trains on the CPU; USE_DL = False skips it)")
        torch.manual_seed(SEED)

        def _bag(X, rows):
            """EmbeddingBag inputs (flat token ids, offsets, bag sizes) for some rows of a CSR token matrix."""
            st = X.indptr[rows].astype(np.int64); ln = X.indptr[rows + 1].astype(np.int64) - st
            off = np.r_[0, np.cumsum(ln)[:-1]].astype(np.int64)
            ix = np.repeat(st - off, ln) + np.arange(int(ln.sum()))
            return (torch.from_numpy(X.indices[ix].astype(np.int64)).to(DL_DEVICE),
                    torch.from_numpy(off).to(DL_DEVICE), torch.from_numpy(ln.astype(np.float32)).to(DL_DEVICE))

        class DLNet(torch.nn.Module):
            """Two token bags (name, address) → MLP → L2-normalized vector. One encoder for S1 and S2/S3 records."""
            def __init__(self):
                super().__init__()
                self.nb = torch.nn.EmbeddingBag(DL_BUCKETS, DL_BAG, mode="sum", sparse=True)
                self.ab = torch.nn.EmbeddingBag(DL_BUCKETS, DL_BAG, mode="sum", sparse=True)
                for e in (self.nb, self.ab): torch.nn.init.normal_(e.weight, 0.0, 0.1)
                self.ln_n, self.ln_a = torch.nn.LayerNorm(DL_BAG), torch.nn.LayerNorm(DL_BAG)
                self.mlp = torch.nn.Sequential(torch.nn.Linear(2 * DL_BAG + 2, 512), torch.nn.GELU(),
                                               torch.nn.Linear(512, 512), torch.nn.GELU(), torch.nn.Linear(512, DL_DIM))
                self.logit_scale = torch.nn.Parameter(torch.tensor(float(np.log(20.0))))   # temperature 0.05, learned
            def forward(self, ni, no, nl, ai, ao, al):
                n = self.ln_n(self.nb(ni, no) / nl.clamp(min=1).sqrt()[:, None])
                a = self.ln_a(self.ab(ai, ao) / al.clamp(min=1).sqrt()[:, None]) * (al > 0).float()[:, None]
                x = torch.cat([n, a, (nl == 0).float()[:, None], (al == 0).float()[:, None]], 1)   # + "empty" flags
                return TF.normalize(self.mlp(x), dim=1)

        def dl_embed(net, Xn, Xa, rows):
            return net(*_bag(Xn, rows), *_bag(Xa, rows))

        class DLEncoder:
            """Blocker-compatible: encode('name_core<TAB>addr_n' texts) → L2-normalized DL_DIM float32 vectors."""
            def __init__(self, net): self.net, self.n_components = net, DL_DIM
            @torch.no_grad()
            def encode(self, texts, chunk=500_000, bs=32_768):
                texts = list(texts)
                out = np.zeros((len(texts), DL_DIM), np.float32)
                self.net.eval()
                for a in range(0, len(texts), chunk):
                    part = [s.split("\t", 1) for s in texts[a:a + chunk]]
                    Xn, Xa = dl_hash([p[0] for p in part], [p[1] if len(p) > 1 else "" for p in part])
                    for b in range(0, len(part), bs):
                        rows = np.arange(b, min(b + bs, len(part)))
                        out[a + b:a + b + len(rows)] = dl_embed(self.net, Xn, Xa, rows).float().cpu().numpy()
                    del Xn, Xa
                return out

        # ---- training pairs: S1 records NOT in the GBDT training sample (so the encoder's scores stay honest) ----
        t_dl = time.time(); tic("learned encoder: training pairs")
        _e = S1ALL_tr.eid.astype(str)
        _pool = S1ALL_tr[(~_e.isin(set(Q_tr.eid.astype(str)))) & _e.isin(set(gt_dict))]
        _pool = _pool.sample(min(DL_POOL_S1, len(_pool)), random_state=SEED + 1).reset_index(drop=True)
        _pe = _pool.eid.astype(str).tolist()
        _pq = np.repeat(np.arange(len(_pe)), [len(gt_dict[e]) for e in _pe])
        _pt = pd.Index(T_tr.eid.astype(str)).get_indexer([x for e in _pe for x in gt_dict[e]])
        _pq, _pt = _pq[_pt >= 0], _pt[_pt >= 0]
        _ut, _ptl = np.unique(_pt, return_inverse=True)          # S2/S3 records used → local row ids
        QN, QA = dl_hash(_pool.name_core.astype(str), _pool.addr_n.astype(str))
        _T = T_tr.iloc[_ut]; TN, TA = dl_hash(_T.name_core.astype(str), _T.addr_n.astype(str)); del _T
        tic(f"  {len(_pool):,} pool S1 | {len(_pq):,} true pairs | hashed {QN.nnz + QA.nnz + TN.nnz + TA.nnz:,} tokens")

        # batch orders: random, and grouped by the S1 name's first word / by its place word, so the other records in a
        # batch are look-alikes (same first word, same locality) → hard in-batch negatives, like the real crowd
        _generic = set(ADDR_ABBR.values()) | set(STATE_MAP.values()) | {"near", "opposite", "floor", "building", "house"}
        _c = _pool.country_key.astype(str).to_numpy(dtype=object)
        _w1 = _pool.name_core.astype(str).map(lambda s: s.split(" ", 1)[0]).to_numpy(dtype=object)
        _loc = _pool.addr_n.astype(str).map(lambda s: next(
            (w for w in reversed(s.split()) if len(w) >= 4 and w.isalpha() and w not in _generic), "")).to_numpy(dtype=object)
        _keys = [None, pd.factorize(_c + "|" + _w1)[0][_pq], pd.factorize(_c + "|" + _loc)[0][_pq]]
        _va = (_pq % 33) == 0                                     # ~3% of the pool S1 held out for a check
        _tri, _vai = np.flatnonzero(~_va), np.flatnonzero(_va)

        def make_batches(idx, key, rng):
            o = rng.permutation(idx) if key is None else idx[np.lexsort((rng.random(len(idx)), key[idx]))]
            return [o[i:i + DL_BATCH] for i in range(0, len(o) - 255, DL_BATCH)]   # drops a tail under 256

        _rs = np.random.default_rng(SEED)
        plan = []
        for ep in range(DL_EPOCHS):
            bb = [b for k in _keys for b in make_batches(_tri, k, _rs)]
            plan += [bb[i] for i in _rs.permutation(len(bb))]
        steps = len(plan)

        net = DLNet().to(DL_DEVICE)
        opt_s = torch.optim.SparseAdam(list(net.nb.parameters()) + list(net.ab.parameters()), lr=DL_LR)
        opt_d = torch.optim.AdamW([p_ for n_, p_ in net.named_parameters() if not n_.startswith(("nb.", "ab."))],
                                  lr=DL_LR, weight_decay=1e-4)
        lr_at = lambda s: DL_LR * min(1.0, (s + 1) / max(1, int(0.03 * steps))) * max(0.05, 1 - s / steps)
        tic(f"learned encoder: training, {steps:,} steps of {DL_BATCH:,} pairs on {DL_DEVICE}")
        net.train(); _run = []
        for s, b in enumerate(plan):
            for o in (opt_s, opt_d):
                for g in o.param_groups: g["lr"] = lr_at(s)
            za = dl_embed(net, QN, QA, _pq[b]); zb = dl_embed(net, TN, TA, _ptl[b])
            sim = net.logit_scale.exp().clamp(max=100) * (za @ zb.T)
            qb = torch.from_numpy(_pq[b]).to(DL_DEVICE)
            same = (qb[:, None] == qb[None, :]) & ~torch.eye(len(b), dtype=torch.bool, device=DL_DEVICE)
            sim = sim.masked_fill(same, float("-inf"))           # other copies of the same S1 are not negatives
            y = torch.arange(len(b), device=DL_DEVICE)
            loss = 0.5 * (TF.cross_entropy(sim, y) + TF.cross_entropy(sim.T, y))
            opt_s.zero_grad(); opt_d.zero_grad(); loss.backward(); opt_s.step(); opt_d.step()
            _run.append(loss.item())
            if (s + 1) % 500 == 0 or s + 1 == steps:
                tic(f"  step {s + 1:,}/{steps:,}  loss {np.mean(_run[-200:]):.4f}  "
                    f"temperature {1 / net.logit_scale.exp().item():.3f} | {gpu_mem()}")

        net.eval(); _acc = []
        with torch.no_grad():
            for b in make_batches(_vai, _keys[1], _rs)[:40]:
                za = dl_embed(net, QN, QA, _pq[b]); zb = dl_embed(net, TN, TA, _ptl[b])
                qb = torch.from_numpy(_pq[b]).to(DL_DEVICE)
                _acc.append((qb[(za @ zb.T).argmax(1)] == qb).float().mean().item())
        if _acc:
            print(f"  held-out pool S1: the most similar of up to {DL_BATCH:,} same-first-word S2/S3 records is a true "
                  f"copy in {np.mean(_acc) * 100:.1f}% of cases")
        for p_ in net.parameters(): p_.requires_grad_(False)
        DL_ENC = DLEncoder(net)
        print(f"learned encoder ready in {time.time() - t_dl:.0f}s: {DL_DIM}-d vectors | {gpu_mem()}")
    except Exception as e:
        DL_ENC = None
        print("learned encoder failed, continuing without it:", repr(e)[:300])
    for _v in ["QN", "QA", "TN", "TA", "opt_s", "opt_d", "plan", "bb", "net", "za", "zb", "sim", "same", "loss", "qb",
               "_pool", "_pe", "_pq", "_pt", "_ptl", "_ut", "_keys", "_c", "_w1", "_loc", "_e", "_run", "_acc"]:
        globals().pop(_v, None)
    if not USE_CE: globals().pop("gt_dict", None)
    gc.collect()
    if GPU_KNN: torch.cuda.empty_cache()
else:
    print("learned encoder off (USE_DL = False)")


## 6. Scalable blocking
| Blocker | How | Catches |
| --- | --- | --- |
| ANN name | char 3–4-gram TF-IDF → 96-d SVD vector → exact kNN (GPU) or FAISS IVF-PQ top-K | typos, reordering, transliteration |
| ANN full | same on core name + address | changed names with the same address |
| ANN address | same on the address alone | same place, changed or made-up name |
| **Learned kNN (v6)** | kNN on the vectors of the learned encoder (5b) | garbled names the SVD vectors blur |
| **Reverse kNN (v6)** | every S2/S3 record searches ALL S1 of its country and keeps its 3 closest | copies crowded out of their S1's own top-K (common name, empty address) |
| Rare token | sparse inverted index of name tokens appearing ≤ 50 times (+ name without spaces) | distinctive words, website names |
| Address key | "housenumber_streetword" and PIN/postal keys appearing ≤ 50 times | totally different name, same address |
| Name + place key | rare pairs of a name word and a place word | common name, same locality |
| Phonetic | sound-alike keys of name words and word pairs | Indian transliteration variants |
| Second hop | neighbours of each S1's best full-ANN match | badly garbled copies that look like a cleaner copy |

On a GPU runtime the ANN blockers search **exactly** in PyTorch (no IVF-PQ compression, no index training).
Everything runs per source (S2, S3) and per country. Indexes are built once; S1 is then queried in chunks.
The reverse kNN searches the FULL S1 table (all train S1, all test S1), so train sees the same crowd as test.


In [ ]:
import faiss
from scipy import sparse
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.decomposition import TruncatedSVD
faiss.omp_set_num_threads(os.cpu_count() or 2)

# defaults, used only if the config cell does not define them
USE_PHON = globals().get("USE_PHON", True); K_PHON = globals().get("K_PHON", 8)
USE_HOP = globals().get("USE_HOP", True); K_HOP = globals().get("K_HOP", 8)
HOP_SEED_MIN = globals().get("HOP_SEED_MIN", 0.6)
USE_REV = globals().get("USE_REV", False); K_REV = globals().get("K_REV", 3)
REV_FIELDS = globals().get("REV_FIELDS", ("full", "dl"))
K_DL = globals().get("K_DL", 20); DL_ENC = globals().get("DL_ENC", None)
KNN_SCORE_ELEMS = globals().get("KNN_SCORE_ELEMS", 2e9)
PHON_MAX_DF = 100            # phonetic keys merge more words, so allow a higher count

def full_text(df):
    return (df["name_core"].astype(str) + " " + df["addr_n"].astype(str)).str.strip()

def field_text(df, field):
    """Text each kNN blocker embeds: the name, the address, or both."""
    if field == "name": return df["name_core"].astype(str)
    if field == "addr": return df["addr_n"].astype(str)
    if field == "dl": return df["name_core"].astype(str) + "\t" + df["addr_n"].astype(str)   # learned encoder (5b)
    return full_text(df)

def name_addr_keys(s):
    """'name word|place word' keys (quartz|crittenden, azamgarh|kumar): each word is common, the pair is rare, so
    copies whose name is common or changed but whose locality is the same still share a rare key."""
    n, _, a = s.partition("\t")
    nt = list(dict.fromkeys(w for w in n.split() if len(w) >= 3 or (w.isdigit() and len(w) >= 2)))[:4]
    at = list(dict.fromkeys(w for w in a.split()                  # place words + house / plot / PIN numbers
                            if (len(w) >= 4 and w.isalpha()) or (w.isdigit() and 2 <= len(w) <= 6)))[:14]
    return [x + "|" + y for x in nt for y in at]

def gpu_svd_components(X, k, n_iter=4, seed=SEED):
    """Top-k right singular vectors of a sparse matrix (same randomized range finder as sklearn's TruncatedSVD),
    computed on the GPU. Returns a (k, n_features) float32 array, like TruncatedSVD.components_."""
    C = X.tocoo()
    idx = torch.from_numpy(np.vstack([C.row, C.col]).astype(np.int64))
    A = torch.sparse_coo_tensor(idx, torch.from_numpy(C.data.astype(np.float32)), C.shape).coalesce().to(KNN_DEVICE)
    At = A.t().coalesce()
    g = torch.Generator(device=KNN_DEVICE).manual_seed(seed)
    Y = torch.sparse.mm(A, torch.randn(C.shape[1], k + 10, device=KNN_DEVICE, generator=g))
    for _ in range(n_iter):                                   # power iterations
        Y = torch.sparse.mm(A, torch.linalg.qr(torch.sparse.mm(At, torch.linalg.qr(Y)[0]))[0])
    Q = torch.linalg.qr(Y)[0]                                 # (m, k+10) orthonormal basis of A's range
    B = torch.sparse.mm(At, Q).T                              # (k+10, n) = Q^T A
    Vt = torch.linalg.svd(B, full_matrices=False)[2][:k]
    out = Vt.float().cpu().numpy()
    del A, At, Y, Q, B, Vt
    if KNN_DEVICE == "cuda": torch.cuda.empty_cache()
    return out

class Embedder:
    """Hashing char n-grams → TF-IDF → SVD → L2-normalized dense vectors (fitted on a sample; no vocab in RAM).
    The SVD is fitted on the GPU when there is one (seconds), with sklearn on the CPU otherwise."""
    def __init__(self, sample_texts, dim=SVD_DIM):
        t0 = time.time()
        self.hv = HashingVectorizer(analyzer="char_wb", ngram_range=(3, 4), n_features=2**18, alternate_sign=False,
                                    norm=None, dtype=np.float32)
        X = par_map(self.hv.transform, sample_texts, min_chunk=50_000)
        self.tf = TfidfTransformer(sublinear_tf=True).fit(X)
        X = self.tf.transform(X)
        self.n_components, self.components, where = dim, None, "CPU"
        if GPU_KNN:
            try: self.components, where = gpu_svd_components(X, dim), "GPU"
            except Exception as e: print("   GPU SVD failed, using the CPU:", repr(e)[:150])
        if self.components is None:
            self.components = TruncatedSVD(dim, algorithm="randomized", n_iter=4,
                                           random_state=SEED).fit(X).components_.astype(np.float32)
        print(f"   embedder fitted in {time.time() - t0:.0f}s (SVD on {where})", flush=True)
    def _encode(self, texts):     # sparse @ dense only: no BLAS, no GPU (also runs in the worker processes)
        Z = np.asarray(self.tf.transform(self.hv.transform(texts)) @ self.components.T, dtype=np.float32)
        n = np.linalg.norm(Z, axis=1, keepdims=True); n[n == 0] = 1
        return Z / n
    def encode(self, texts, chunk=200_000):
        texts = list(texts)
        if not texts: return np.zeros((0, self.n_components), np.float32)
        if N_PROC > 1: return par_map(self._encode, texts, min_chunk=50_000)      # all CPU cores
        return np.concatenate([self._encode(texts[a:a + chunk]) for a in range(0, len(texts), chunk)])

class TorchFlatIP:
    """Exact inner-product kNN on the GPU (PyTorch), with the same .search(Q, k) -> (D, I) interface as a FAISS index.
    Vectors stay on the GPU in float16 (10M x 96 = 2 GB); queries (numpy or a GPU tensor) go in batches of
    KNN_SCORE_ELEMS scores. Blocker.offload() can park the vectors in CPU RAM; the next search moves them back."""
    def __init__(self, V):
        self.X = torch.from_numpy(np.ascontiguousarray(V)).to(KNN_DEVICE, torch.float16)
        self.ntotal = len(V)
    def search(self, Q, k):
        if self.X.device.type != KNN_DEVICE: self.X = self.X.to(KNN_DEVICE)
        k = min(k, self.ntotal)
        bs = int(np.clip(KNN_SCORE_ELEMS / max(self.ntotal, 1), 256, 16384))
        D = np.empty((len(Q), k), np.float32); I = np.empty((len(Q), k), np.int64)
        for a in range(0, len(Q), bs):
            q = Q[a:a + bs]
            q = (q if torch.is_tensor(q) else torch.from_numpy(np.ascontiguousarray(q))).to(KNN_DEVICE, torch.float16)
            s, i = torch.topk(q @ self.X.T, k, dim=1)
            D[a:a + bs] = s.float().cpu().numpy(); I[a:a + bs] = i.cpu().numpy()
        return D, I

def build_index(V):
    """GPU: exact search. CPU: exact index for small groups, IVF-PQ (compressed, approximate) for big ones."""
    global GPU_KNN
    if GPU_KNN:
        try: return TorchFlatIP(V)
        except Exception as e:
            GPU_KNN = False; print("  GPU index failed, switching to FAISS on CPU:", repr(e)[:200])
    n, d = V.shape
    if n < IVF_MIN:
        idx = faiss.IndexFlatIP(d); idx.add(V); return idx
    nlist = int(min(4096, max(64, 2 * np.sqrt(n))))
    idx = faiss.IndexIVFPQ(faiss.IndexFlatIP(d), d, nlist, PQ_M, 8, faiss.METRIC_INNER_PRODUCT)
    samp = V[np.random.default_rng(SEED).choice(n, min(n, max(nlist * 40, 10_000)), replace=False)]
    idx.train(samp); idx.add(V); idx.nprobe = NPROBE
    return idx

def csr_topk(R, k):
    """Vectorized top-k per row of a sparse score matrix → (row, col, score, rank)."""
    R = R.tocsr(); R.eliminate_zeros()
    rows = np.repeat(np.arange(R.shape[0], dtype=np.int64), np.diff(R.indptr))
    if len(rows) == 0: return rows, rows, np.zeros(0, np.float32), rows
    order = np.lexsort((-R.data, rows))
    rows, cols, data = rows[order], R.indices[order], R.data[order]
    rank = np.arange(len(rows)) - R.indptr[:-1][rows]
    m = rank < k
    return rows[m], cols[m], data[m].astype(np.float32), (rank[m] + 1)

def name_tokens(s):
    t = [w for w in s.split() if len(w) >= 3]
    ns = s.replace(" ", "")
    return t + (["NS" + ns] if len(ns) >= 6 else [])

def addr_keys(s):
    toks = s.split()
    nums = [t for t in toks if t.isdigit()]
    words = [t for t in toks if len(t) >= 4 and t.isalpha()]
    keys = [nums[0] + "_" + w for w in words] if nums else []
    if len(nums) >= 2: keys.append("S" + "_".join(nums[:3]))        # house-number sequence: "19_1_16"
    return keys + ["P" + t for t in nums if 5 <= len(t) <= 6]

# ---- phonetic keys (phon() is defined in section 5) ----
def phon_tokens(s):
    """Single-word keys (F...) plus two-word keys (G...); two-word keys stay rare even when words are common."""
    t = [phon(w) for w in s.split() if len(w) >= 3 and w.isalpha()]
    t = [w for w in t if len(w) >= 3]
    return ["F" + w for w in t] + ["G" + t[i] + "_" + t[i + 1] for i in range(len(t) - 1)]

class RareIndex:
    """Sparse inverted index over rare tokens: score(q, t) = sum of IDF of rare tokens they share."""
    def __init__(self, texts, analyzer, max_df=RARE_MAX_DF):
        self.hv = HashingVectorizer(analyzer=analyzer, n_features=2**22, binary=True, norm=None,
                                    alternate_sign=False, dtype=np.float32)
        T = par_map(self.hv.transform, texts, min_chunk=50_000).tocsc()
        df = np.diff(T.indptr); n = T.shape[0]
        keep = (df >= 1) & (df <= max_df)
        self.idf = np.where(keep, np.log(n / np.maximum(df, 1)), 0).astype(np.float32)
        T = sparse.csc_matrix(T.multiply(keep.astype(np.float32)[None, :]))   # zero-out common tokens
        T.eliminate_zeros()
        self.TT = T.T.tocsr()
    def _query(self, texts, k):
        Qw = self.hv.transform(texts).multiply(self.idf[None, :]).tocsr()
        return csr_topk(Qw @ self.TT, k)
    def query(self, texts, k):
        """The (queries x targets) sparse product is the costly part → queries are split over the CPU cores."""
        if N_PROC < 2 or len(texts) < 20_000: return self._query(texts, k)
        def part(d):
            r, c, s, rk = self._query(d.s.tolist(), k)
            return pd.DataFrame({"r": d.i.values[r], "c": c, "s": s, "rk": rk})
        R = par_map(part, pd.DataFrame({"i": np.arange(len(texts)), "s": list(texts)}), min_chunk=5_000)
        return R.r.values, R.c.values, R.s.values, R.rk.values

class Blocker:
    """Built once per stage (train / test). `candidates(qdf)` returns the candidate frame for a chunk of S1."""
    def __init__(self, Q, T, sample=300_000):
        self.T = T
        self.t_country = T.country_key.astype(str).to_numpy(dtype=object)   # computed once, reused per chunk
        rs = np.random.default_rng(SEED)
        tidx = rs.choice(len(T), min(sample, len(T)), replace=False)
        qidx = rs.choice(len(Q), min(sample // 3, len(Q)), replace=False)
        tic("  fitting embedders")
        self.emb = {f: Embedder(pd.concat([field_text(T.iloc[tidx], f), field_text(Q.iloc[qidx], f)]).tolist())
                    for f in (("name", "full", "addr") if K_ADDR else ("name", "full"))}
        if DL_ENC is not None: self.emb["dl"] = DL_ENC      # learned retrieval encoder (5b)
        self.groups = {}   # (src, country) -> global target ids
        for (src, c), g in T.groupby(["src", "country_key"], observed=True).indices.items():
            self.groups[(int(src), str(c))] = np.asarray(g, np.int64)
        self.idx, self.dl_T, self.rev_s1, self.t_best = {}, None, None, {}
        if "dl" in self.emb: self.dl_T = np.zeros((len(T), DL_DIM), np.float16)   # for dl_cos of every candidate pair
        for field in self.emb:
            for key, gids in self.groups.items():
                tic(f"  index {field} {key}: {len(gids):,}")
                sub = T.iloc[gids]
                texts = field_text(sub, field).tolist()
                V = self.emb[field].encode(texts)
                if field == "dl": self.dl_T[gids] = V
                self.idx[(field,) + key] = build_index(V); del V; gc.collect()
        self.rare, self.akey, self.phon, self.nakey = {}, {}, {}, {}
        for src in (2, 3):
            m = np.flatnonzero(T.src.values == src)
            tic(f"  rare-token + address-key{' + phonetic' if USE_PHON else ''} index S{src}: {len(m):,}")
            names = T.name_core.iloc[m].astype(str).tolist()
            self.rare[src] = (m, RareIndex(names, name_tokens))
            self.akey[src] = (m, RareIndex(T.addr_n.iloc[m].astype(str).tolist(), addr_keys))
            if USE_PHON:
                self.phon[src] = (m, RareIndex(names, phon_tokens, max_df=PHON_MAX_DF))
            if K_NAKEY:
                na = (T.name_core.iloc[m].astype(str) + "\t" + T.addr_n.iloc[m].astype(str)).tolist()
                self.nakey[src] = (m, RareIndex(na, name_addr_keys)); del na
            del names; gc.collect()

    def set_nprobe(self, n):
        """Change IVF search width on the built indexes (no rebuild needed)."""
        for ix in self.idx.values():
            if hasattr(ix, "nprobe"): ix.nprobe = n

    def offload(self):
        """Park the GPU kNN vectors in CPU RAM (frees GPU memory for model training); the next search moves them back."""
        for ix in self.idx.values():
            if isinstance(ix, TorchFlatIP): ix.X = ix.X.cpu()
        if GPU_KNN: torch.cuda.empty_cache()

    def _group_vectors(self, field, key):
        """Target vectors of one (src, country) group: the GPU index's own copy when possible, else re-encoded."""
        ix = self.idx[(field,) + key]
        if isinstance(ix, TorchFlatIP): return ix.X
        if field == "dl" and self.dl_T is not None: return self.dl_T[self.groups[key]].astype(np.float32)
        return self.emb[field].encode(field_text(self.T.iloc[self.groups[key]], field).tolist())

    def build_reverse(self, S1ALL):
        """Reverse kNN: every S2/S3 record searches ALL S1 records of its country (REV_FIELDS vectors) and keeps its K_REV
        closest S1. A copy crowded out of its S1's own top-K (common name, empty address) is usually still closest to the
        right S1 from its own side. Stored per S1 (CSR-like, by entity id), so candidates() only looks pairs up.
        Also keeps each S2/S3 record's similarity to its closest S1 (t_best), for the dl_t_gap feature."""
        t0 = time.time()
        try:
            s1c = S1ALL.country_key.astype(str).values
            nT = len(self.T); S, Tt, D, R = [], [], [], []
            for field in [f for f in REV_FIELDS if f in self.emb]:
                V1 = self.emb[field].encode(field_text(S1ALL, field).tolist())
                best = np.zeros(nT, np.float16)
                for c in np.unique(s1c):
                    keys = [k for k in self.groups if k[1] == c]
                    gi = np.flatnonzero(s1c == c)
                    if not keys or len(gi) == 0: continue
                    ix = build_index(V1[gi])
                    for key in keys:
                        gids = self.groups[key]
                        Dk, Ik = ix.search(self._group_vectors(field, key), min(K_REV, len(gi)))
                        ok = Ik >= 0
                        S.append(gi[Ik[ok]].astype(np.int32)); Tt.append(np.repeat(gids, ok.sum(1)).astype(np.int32))
                        D.append(Dk[ok].astype(np.float16))
                        R.append(np.broadcast_to(np.arange(1, Ik.shape[1] + 1, dtype=np.int8), Ik.shape)[ok])
                        best[gids] = np.where(ok[:, 0], Dk[:, 0], 0)
                        del Dk, Ik, ok
                    del ix; gc.collect()
                    if GPU_KNN: torch.cuda.empty_cache()
                self.t_best[field] = best; del V1
                tic(f"  reverse kNN on {field} vectors done | {sum(map(len, S)):,} pairs so far | {gpu_mem()}")
            s1 = np.concatenate(S); t = np.concatenate(Tt); d = np.concatenate(D); r = np.concatenate(R)
            del S, Tt, D, R
            key = s1.astype(np.int64) * nT + t; del s1, t
            o = np.argsort(key, kind="stable"); key = key[o]
            st = np.flatnonzero(np.r_[True, key[1:] != key[:-1]])          # one row per (S1, S2/S3) pair
            self.rev_s1 = (key[st] // nT).astype(np.int32); self.rev_t = (key[st] % nT).astype(np.int32)
            self.rev_score = np.maximum.reduceat(d[o].astype(np.float32), st).astype(np.float16)
            self.rev_rank = np.minimum.reduceat(r[o], st).astype(np.int8)
            self.rev_nf = np.diff(np.r_[st, len(key)]).astype(np.int8)    # in how many REV_FIELDS it was found
            self.rev_ptr = np.searchsorted(self.rev_s1, np.arange(len(S1ALL) + 1)).astype(np.int64)
            self.rev_key = pd.Index(S1ALL.eid.astype(str))
            del key, o, st, d, r; gc.collect()
            print(f"  reverse kNN: {len(self.rev_s1):,} (S1, S2/S3) pairs = {len(self.rev_s1) / len(S1ALL):.1f} per S1 | "
                  f"{time.time() - t0:.0f}s | {gpu_mem()}")
        except Exception as e:
            self.rev_s1, self.t_best = None, {}
            print("  reverse kNN failed, continuing without it:", repr(e)[:300])

    def _rev(self, qdf):
        """Reverse-kNN pairs of the S1 records in qdf (looked up by entity id)."""
        p = self.rev_key.get_indexer(qdf.eid.astype(str).values)
        ok = np.flatnonzero(p >= 0)
        st = self.rev_ptr[p[ok]]; cnt = self.rev_ptr[p[ok] + 1] - st
        off = np.r_[0, np.cumsum(cnt)[:-1]].astype(np.int64) if len(cnt) else np.zeros(0, np.int64)
        ix = np.repeat(st - off, cnt) + np.arange(int(cnt.sum()))
        return pd.DataFrame({"q": np.repeat(ok, cnt).astype(np.int64), "t": self.rev_t[ix].astype(np.int64),
                             "rev_score": self.rev_score[ix].astype(np.float32), "rev_rank": self.rev_rank[ix],
                             "rev_nf": self.rev_nf[ix]})

    def _rowcos(self, Vq, q, t, block=1_000_000):
        """Learned-encoder cosine of each (S1 row of Vq, S2/S3 record) pair."""
        out = np.empty(len(q), np.float32)
        for a in range(0, len(q), block):
            out[a:a + block] = np.einsum("ij,ij->i", Vq[q[a:a + block]], self.dl_T[t[a:a + block]].astype(np.float32))
        return out

    def _ann(self, qdf, field, k, V=None):
        out = []
        if V is None: V = self.emb[field].encode(field_text(qdf, field).tolist())
        qc = qdf.country_key.astype(str).values
        for src in (2, 3):
            for c in np.unique(qc):
                qi = np.flatnonzero(qc == c)
                keys = [(src, c)] if (src, c) in self.groups and BLOCK_BY_COUNTRY else \
                       [g for g in self.groups if g[0] == src]           # unknown country → search all groups
                res = []
                for key in keys:
                    D, I = self.idx[(field,) + key].search(V[qi], min(k, len(self.groups[key])))
                    ok = I >= 0
                    res.append((np.repeat(qi, ok.sum(1)), self.groups[key][I[ok]], D[ok]))
                r = [np.concatenate(x) for x in zip(*res)]
                df = pd.DataFrame({"q": r[0], "t": r[1], "s": r[2].astype(np.float32)})
                if field == "addr": df = df[df.s >= ADDR_MIN]          # empty / generic addresses match everything
                df = df.sort_values(["q", "s"], ascending=[True, False])
                df["r"] = df.groupby("q").cumcount() + 1
                out.append(df[df.r <= k])
        d = pd.concat(out, ignore_index=True)
        return d.rename(columns={"s": f"ann_{field}_score", "r": f"ann_{field}_rank"})

    def _rare(self, qdf, which, k, name):
        out = []
        if which == "nakey":
            texts = (qdf.name_core.astype(str) + "\t" + qdf.addr_n.astype(str)).tolist()
        else:
            texts = qdf["addr_n" if which == "akey" else "name_core"].astype(str).tolist()
        store = {"rare": self.rare, "akey": self.akey, "phon": self.phon, "nakey": self.nakey}[which]
        for src in (2, 3):
            m, ri = store[src]
            rows, cols, sc, rk = ri.query(texts, k)
            out.append(pd.DataFrame({"q": rows, "t": m[cols], f"{name}_score": sc, f"{name}_rank": rk}))
        d = pd.concat(out, ignore_index=True)
        if BLOCK_BY_COUNTRY and len(d):   # these indexes are global: drop cross-country pairs
            same = qdf.country_key.astype(str).values[d.q.values] == self.t_country[d.t.values]
            d = d[same]
        return d

    def _hop(self, full):
        """Second hop: FAISS neighbours of each S1's best full-ANN match per source (other copies of the business)."""
        empty = pd.DataFrame({"q": np.zeros(0, np.int64), "t": np.zeros(0, np.int64),
                              "hop_score": np.zeros(0, np.float32), "hop_rank": np.zeros(0, np.int64)})
        s = full[(full.ann_full_rank == 1) & (full.ann_full_score >= HOP_SEED_MIN)]
        if len(s) == 0: return empty
        seed_q, seed_t = s.q.values.astype(np.int64), s.t.values.astype(np.int64)
        V = self.emb["full"].encode(full_text(self.T.iloc[seed_t]).tolist())
        sc = self.t_country[seed_t]
        out = []
        for src in (2, 3):
            for c in np.unique(sc):
                key = (src, c)
                if key not in self.groups: continue
                ii = np.flatnonzero(sc == c)
                D, I = self.idx[("full",) + key].search(V[ii], min(K_HOP + 1, len(self.groups[key])))
                ok = I >= 0
                out.append(pd.DataFrame({"q": np.repeat(seed_q[ii], ok.sum(1)),
                                         "t": self.groups[key][I[ok]], "hop_score": D[ok].astype(np.float32)}))
        if not out: return empty
        d = pd.concat(out, ignore_index=True).groupby(["q", "t"], as_index=False).hop_score.max()
        d = d.sort_values(["q", "hop_score"], ascending=[True, False])
        d["hop_rank"] = d.groupby("q").cumcount() + 1
        return d[d.hop_rank <= 2 * K_HOP]

    def candidates(self, qdf):
        Vdl = self.emb["dl"].encode(field_text(qdf, "dl").tolist()) if "dl" in self.emb else None
        full = self._ann(qdf, "full", K_FULL)
        parts = [self._ann(qdf, "name", K_NAME), full,
                 self._rare(qdf, "rare", K_RARE, "rare"), self._rare(qdf, "akey", K_AKEY, "akey")]
        if self.phon:
            parts.append(self._rare(qdf, "phon", K_PHON, "phon"))
        if "addr" in self.emb and K_ADDR:
            parts.append(self._ann(qdf, "addr", K_ADDR))
        if self.nakey and K_NAKEY:
            parts.append(self._rare(qdf, "nakey", K_NAKEY, "nakey"))
        if Vdl is not None and K_DL:
            parts.append(self._ann(qdf, "dl", K_DL, V=Vdl))
        if USE_REV and self.rev_s1 is not None:
            parts.append(self._rev(qdf))
        if USE_HOP:
            parts.append(self._hop(full))
        u = pd.concat([p[["q", "t"]] for p in parts], ignore_index=True).drop_duplicates()
        for p in parts: u = u.merge(p, on=["q", "t"], how="left")
        for c in u.columns:
            if c.endswith("_score"): u[c] = u[c].fillna(0).astype(np.float32)
            if c.endswith("_rank"):  u[c] = u[c].fillna(99).astype(np.int16)
        if "rev_nf" in u: u["rev_nf"] = u["rev_nf"].fillna(0).astype(np.int8)
        u["q"] = u["q"].astype(np.int64); u["t"] = u["t"].astype(np.int64)
        u = u.sort_values(["q", "t"]).reset_index(drop=True)
        if Vdl is not None:   # learned similarity of EVERY candidate pair, not only of the pairs the dl blocker found
            q, t = u.q.values, u.t.values
            u["dl_cos"] = self._rowcos(Vdl, q, t)
            if "dl" in self.t_best:   # 0 = this S1 is the copy's closest S1 among all S1 of the country
                u["dl_t_gap"] = (self.t_best["dl"][t].astype(np.float32) - u.dl_cos.values).astype(np.float32)
            if "nn_name_core" in qdf and "nn_cos" in qdf:   # is the copy closer to this S1 than to its look-alike S1?
                Vn = self.emb["dl"].encode((qdf.nn_name_core.astype(str) + "\t" + qdf.nn_addr_n.astype(str)).tolist())
                has = qdf.nn_cos.values[q] > 0
                u["dl_nn_margin"] = np.where(has, u.dl_cos.values - self._rowcos(Vn, q, t), 0).astype(np.float32)
        return u

def add_s1_neighbours(Q, S1ALL, emb):
    """For every S1 in Q: its most similar OTHER S1 record of the same country, searched in the FULL S1 table (train and
    test see the same crowd). Its name / address / numbers are stored as nn_* columns; make_features compares each
    candidate with both this S1 and its look-alike ('is the copy closer to me or to it?')."""
    V = emb.encode(full_text(S1ALL).tolist())
    me = pd.Index(S1ALL.eid.astype(str)).get_indexer(Q.eid.astype(str))
    QV = V[np.maximum(me, 0)]
    if (me < 0).any(): QV[me < 0] = emb.encode(full_text(Q[me < 0]).tolist())
    ca, cq = S1ALL.country_key.astype(str).values, Q.country_key.astype(str).values
    nn, cos = np.full(len(Q), -1, np.int64), np.zeros(len(Q), np.float32)
    for c in np.unique(cq):
        gi, qi = np.flatnonzero(ca == c), np.flatnonzero(cq == c)
        if len(gi) < 2: continue
        D, I = build_index(V[gi]).search(QV[qi], 2)
        g = np.where(I >= 0, gi[np.maximum(I, 0)], -1)
        pick = (g[:, 0] == me[qi]).astype(np.int64)                  # skip the S1 itself (also for exact ties)
        r = np.arange(len(qi))
        nn[qi], cos[qi] = g[r, pick], D[r, pick]
    ok = nn >= 0
    for col in ("name_core", "addr_n", "nums"):
        v = S1ALL[col].astype(str).to_numpy(dtype=object)[np.maximum(nn, 0)]
        Q["nn_" + col] = pd.Series(np.where(ok, v, ""), index=Q.index).astype("string[pyarrow]")
    Q["nn_cos"] = np.where(ok, cos, 0).astype(np.float32)
    print(f"  look-alikes: median cosine {np.median(Q.nn_cos):.2f} | S1 with a look-alike above 0.9: "
          f"{(Q.nn_cos > 0.9).mean()*100:.1f}% | {gpu_mem()}")
    del V, QV; gc.collect()

def add_name_counts(Q, T, S1ALL):
    """How many S1 records (full S1 table, same country) and how many S2/S3 records share each record's core name.
    For a copy with an empty address, the S1 count is the number of S1 records it could belong to."""
    key = lambda d: d.country_key.astype(str) + "|" + d.name_core.astype(str)
    s1c, tc = key(S1ALL).value_counts(), key(T).value_counts()
    for d in (Q, T):
        k = key(d)
        d["s1_same_name"] = k.map(s1c).fillna(0).astype(np.float32).values
        d["t_same_name"] = k.map(tc).fillna(0).astype(np.float32).values
    print(f"  name counts: S1 names shared by 2+ S1 records: {(Q.s1_same_name > 1).mean()*100:.1f}% | "
          f"S2/S3 names matching exactly one S1 name: {(T.s1_same_name == 1).mean()*100:.1f}%")

tic("building TRAIN blocker"); BLK_tr = Blocker(Q_tr, T_tr); tic("done")
if USE_REV and "S1ALL_tr" in globals():
    tic("reverse kNN: every S2/S3 record → its closest S1 among ALL train S1 (the same crowd as test)")
    BLK_tr.build_reverse(S1ALL_tr)
    if "dl" in BLK_tr.t_best and "T_OWNED_IDS" in globals():   # reference for the label-free orphan check (section 12)
        _own = np.zeros(len(T_tr), bool); _ix = pd.Index(T_tr.eid.astype(str)).get_indexer(T_OWNED_IDS)
        _own[_ix[_ix >= 0]] = True
        _tb = BLK_tr.t_best["dl"].astype(np.float32)
        TB_BINS = np.linspace(-1, 1, 201)
        TB_REF = dict(own=np.histogram(_tb[_own], TB_BINS, density=True)[0],
                      orph=np.histogram(_tb[~_own], TB_BINS, density=True)[0], share=float(1 - _own.mean()))
        print(f"  orphan S2/S3 records (no S1 record) in train: {TB_REF['share'] * 100:.1f}% | similarity to the closest "
              f"S1: median {np.median(_tb[_own]):.3f} with an owner vs {np.median(_tb[~_own]):.3f} for orphans")
        del _own, _ix, _tb
globals().pop("T_OWNED_IDS", None)
if USE_S1_NN:
    tic("S1 look-alikes (train sample vs ALL train S1)"); add_s1_neighbours(Q_tr, S1ALL_tr, BLK_tr.emb["full"])
if "S1ALL_tr" in globals(): tic("S1 / S2-S3 name counts (train)"); add_name_counts(Q_tr, T_tr, S1ALL_tr)
if not USE_CE: globals().pop("S1ALL_tr", None); gc.collect()   # the cross-encoder pool (10a) is drawn from it

### 6b. Blocking recall check (a few minutes, no features or model)
Measures how many true pairs of 30k training S1 the blocker finds, and the **F0.5 ceiling** (the macro F0.5 of a perfect
model on these candidates), with and without the v6 blockers, plus the recall of each blocker alone.
With `AUTO_BLOCKING = True` it keeps the cheapest option unless a bigger one adds at least `MIN_RECALL_GAIN` recall.


In [ ]:
if RUN_RECALL_CHECK:
    QS = Q_tr.iloc[:RECALL_CHECK_N]
    t_of = pd.Index(T_tr.eid.astype(str))
    pairs = [(qi, x) for qi, e in enumerate(QS.eid.astype(str)) for x in gt.get(e, [])]
    TRUE = pd.DataFrame({"q": np.array([p[0] for p in pairs], np.int64),
                         "t": t_of.get_indexer([p[1] for p in pairs]).astype(np.int64)})
    TRUE = TRUE[TRUE.t >= 0].drop_duplicates().reset_index(drop=True)
    TRUE["country"] = QS.country_key.astype(str).values[TRUE.q.values]
    nt_q = np.bincount(TRUE.q.values, minlength=len(QS))          # true matches per S1 (0 = singleton)

    def recall_run(nprobe, kn, kf, hop, label, v6=True):
        """One blocking setup on QS → pair recall, F0.5 ceiling, candidates per S1, recall of each blocker alone.
        v6=False switches the reverse kNN and the learned-encoder kNN off (reference)."""
        global K_NAME, K_FULL, USE_HOP, USE_REV, K_DL
        K_NAME, K_FULL, USE_HOP = kn, kf, hop
        saved = (USE_REV, K_DL)
        if not v6: USE_REV, K_DL = False, 0
        BLK_tr.set_nprobe(nprobe)
        try:
            t0 = time.time(); c = BLK_tr.candidates(QS); sec = time.time() - t0
        finally:
            USE_REV, K_DL = saved
        m = TRUE.merge(c, on=["q", "t"], how="left", indicator=True)
        hit = (m["_merge"] == "both").values
        by_c = TRUE.assign(h=hit).groupby("country").h.mean().round(4).to_dict()
        hq = np.bincount(TRUE.q.values[hit], minlength=len(QS))
        with np.errstate(divide="ignore", invalid="ignore"):
            ceil = float(np.where(nt_q == 0, 1.0, np.where(hq > 0, 1.25 * (hq / nt_q) / (0.25 + hq / nt_q), 0)).mean())
        res = dict(label=label, K_NAME=kn, K_FULL=kf, USE_HOP=hop, recall=float(hit.mean()), ceiling=ceil,
                   cands=len(c) / len(QS))
        print(f"{label:34s} nprobe={nprobe:3d} K_NAME={kn:2d} K_FULL={kf:2d} hop={hop!s:5s} | recall {res['recall']:.4f} "
              f"{by_c} | ceiling {ceil:.4f} | cands/S1 {res['cands']:.1f} | {sec:.0f}s", flush=True)
        alone = {b[:-5]: round(float((m[b] < 99).mean()), 4) for b in m.columns if b.endswith("_rank")}
        print(f"{'':34s} recall of each blocker alone: {alone}", flush=True)
        CANDS[label] = c[["q", "t"]]; del c, m; gc.collect()     # kept for the "what is missed" report below
        return res

    cfg_k, cfg_hop = (K_NAME, K_FULL), USE_HOP
    CANDS = {}
    V6 = bool((USE_REV and BLK_tr.rev_s1 is not None) or ("dl" in BLK_tr.emb and K_DL))
    if GPU_KNN:   # exact search: nprobe changes nothing, and hop never paid off (+0.1 pts for 2x time)
        runs = [recall_run(NPROBE, *cfg_k, False, "config without v6 (reference)", v6=False) if V6 else None,
                recall_run(NPROBE, *cfg_k, False, "config + reverse kNN + learned" if V6 else "config, no hop")]
        if tuple(BIG_K) != tuple(cfg_k): runs.append(recall_run(NPROBE, *BIG_K, False, "bigger K"))
    else:
        runs = [recall_run(NPROBE, *cfg_k, False, "config without v6 (reference)", v6=False) if V6 else
                recall_run(12, 12, 12, False, "old setup (reference)"),
                recall_run(NPROBE, *cfg_k, False, "config, no hop"),
                recall_run(NPROBE, *cfg_k, True, "config + hop")]
        if tuple(BIG_K) != tuple(cfg_k):
            runs += [recall_run(NPROBE, *BIG_K, False, "bigger K, no hop"), recall_run(NPROBE, *BIG_K, True, "bigger K + hop")]
    BLK_tr.set_nprobe(NPROBE)

    if AUTO_BLOCKING:
        options = sorted([r for r in runs[1:] if r["cands"] <= MAX_CANDS_PER_S1], key=lambda r: r["cands"])
        best = options[0] if options else runs[1]
        for r in options[1:]:
            if r["recall"] >= best["recall"] + MIN_RECALL_GAIN: best = r
        K_NAME, K_FULL, USE_HOP = best["K_NAME"], best["K_FULL"], best["USE_HOP"]
        print(f"\nAUTO → {best['label']}: K_NAME={K_NAME} K_FULL={K_FULL} USE_HOP={USE_HOP} "
              f"| recall {best['recall']:.4f} | ceiling {best['ceiling']:.4f} | {best['cands']:.1f} cands/S1")
    else:
        K_NAME, K_FULL = cfg_k; USE_HOP = cfg_hop
        print(f"\nkeeping config: K_NAME={K_NAME} K_FULL={K_FULL} USE_HOP={USE_HOP}")

    # ---- what the chosen setup still misses (tells us which kind of blocker is missing) ----
    BLK_tr.set_nprobe(NPROBE)
    chosen = best["label"] if AUTO_BLOCKING else runs[1]["label"]
    c = CANDS[chosen] if chosen in CANDS else BLK_tr.candidates(QS)
    hit = TRUE.merge(c[["q", "t"]].assign(f=1), on=["q", "t"], how="left").f.fillna(0).astype(bool).values
    miss = TRUE[~hit]
    ind_t = T_tr.is_indic.values
    print(f"\nmissed {len(miss):,} of {len(TRUE):,} true pairs | by country: {miss.country.value_counts().to_dict()}")
    print(f"Indic-script targets: {ind_t[miss.t.values].mean()*100:.1f}% of missed pairs vs "
          f"{ind_t[TRUE.t.values[hit]].mean()*100:.1f}% of found pairs")
    emp = T_tr.addr_n.astype(str).to_numpy(dtype=object) == ""
    print(f"S2/S3 record has an empty address: {emp[miss.t.values].mean()*100:.1f}% of missed pairs vs "
          f"{emp[TRUE.t.values[hit]].mean()*100:.1f}% of found pairs")
    for r in miss.groupby("country").head(8).itertuples():
        a, b = QS.iloc[r.q], T_tr.iloc[r.t]
        print(f"[{r.country}{' indic' if ind_t[r.t] else ''}] {a.name_core!r} | {a.addr_n!r}\n"
              f"          → {b.name_core!r} | {b.addr_n!r}")
    del c, hit, miss, QS, TRUE, pairs, CANDS, nt_q, emp; gc.collect()


## 7. Pairwise features (vectorized, computed chunk by chunk)

In [ ]:
PW_WORKERS = -1   # RapidFuzz threads (set to 1 inside the worker processes of make_features_fast)
def pw(a, b, scorer):
    return cpdist(a, b, scorer=scorer, workers=PW_WORKERS, dtype=np.float32)

class FeatSpaces:
    """Stateless hashing spaces + IDF weights fitted once per stage on a sample of target records."""
    def __init__(self, T, sample=400_000):
        s = T.iloc[np.random.default_rng(SEED).choice(len(T), min(sample, len(T)), replace=False)]
        H = lambda **kw: HashingVectorizer(n_features=2**20, alternate_sign=False, norm=None, dtype=np.float32, **kw)
        self.sp = {"name_core": H(analyzer="char_wb", ngram_range=(2, 4)), "name_n": H(analyzer="word"),
                   "addr_n": H(analyzer="char_wb", ngram_range=(3, 4)), "full": H(analyzer="char_wb", ngram_range=(3, 4))}
        txt = {"name_core": s.name_core.astype(str), "name_n": s.name_n.astype(str), "addr_n": s.addr_n.astype(str),
               "full": full_text(s)}
        self.tf = {k: TfidfTransformer(sublinear_tf=True).fit(par_map(v.transform, txt[k].tolist(), min_chunk=25_000))
                   for k, v in self.sp.items()}
        self.bin = {"name": H(analyzer="word", binary=True), "addr": H(analyzer="word", binary=True),
                    "num": H(analyzer="word", binary=True, token_pattern=r"\b\d+\b")}
        self.idf = {k: np.asarray(TfidfTransformer().fit(par_map(self.bin[k].transform, txt[c].tolist(),
                                                                  min_chunk=25_000)).idf_, np.float32)
                    for k, c in [("name", "name_core"), ("addr", "addr_n")]}
    def vec(self, key, texts):
        return self.tf[key].transform(par_map(self.sp[key].transform, texts, min_chunk=100_000))

def rowdot(X, Y):
    return np.asarray(X.multiply(Y).sum(1)).ravel().astype(np.float32)

def set_feats(Xa, Xb, w=None):
    """Jaccard (and IDF-weighted Jaccard) between the token sets of paired rows, fully vectorized."""
    if w is not None: Xa, Xb = Xa.multiply(w[None, :]).tocsr(), Xb.multiply(w[None, :]).tocsr()
    inter = np.asarray(Xa.minimum(Xb).sum(1)).ravel()
    tot = np.asarray(Xa.sum(1)).ravel() + np.asarray(Xb.sum(1)).ravel() - inter
    return np.where(tot > 0, inter / np.maximum(tot, 1e-9), 0).astype(np.float32)

def seq_agree(a, b):
    """1 = same house-number sequence, -1 = both have numbers but they differ, 0 = one side has none."""
    return np.array([0 if not x or not y else (1 if x == y else -1) for x, y in zip(a, b)], np.int8)

def num_rel(a, b):
    """Best relation between the address numbers of two records: 3 same number (2+ digits), 2 one contains the other
    (916 / 10916, 195 / 195197), 1 one digit apart (85 / 45) or same single digit, -1 unrelated, 0 one side has none."""
    if not a or not b: return 0
    best = -1
    for x in a.split()[:6]:
        for y in b.split()[:6]:
            if x == y:
                if len(x) >= 2: return 3
                best = max(best, 1)
            elif min(len(x), len(y)) >= 2 and (x.startswith(y) or x.endswith(y) or y.startswith(x) or y.endswith(x)):
                best = 2
            elif len(x) == len(y) >= 2 and sum(p != q for p, q in zip(x, y)) == 1:
                best = max(best, 1)
    return best

def agree(a, b):
    a, b = np.asarray(a, dtype=object), np.asarray(b, dtype=object)
    has = (a != "") & (b != "")
    return np.where(has, np.where(a == b, 1, -1), 0).astype(np.int8)

def make_features(c, Q, T, FS):
    """c: candidate frame (q, t, blocker cols) for ONE chunk of S1. Returns feature frame."""
    F = c.copy()
    uq, iq = np.unique(c.q.values, return_inverse=True)
    ut, it = np.unique(c.t.values, return_inverse=True)
    A, B = Q.iloc[uq].reset_index(drop=True), T.iloc[ut].reset_index(drop=True)
    def L(df, col, ix): return df[col].astype(str).to_numpy(dtype=object)[ix].tolist()
    an, bn = L(A, "name_n", iq), L(B, "name_n", it)
    ac, bc = L(A, "name_core", iq), L(B, "name_core", it)
    aa, ba = L(A, "addr_n", iq), L(B, "addr_n", it)
    F["n_ratio"] = pw(an, bn, fuzz.ratio); F["n_tset"] = pw(an, bn, fuzz.token_set_ratio)
    F["n_tsort"] = pw(an, bn, fuzz.token_sort_ratio); F["n_partial"] = pw(an, bn, fuzz.partial_ratio)
    F["n_jw"] = pw(an, bn, JaroWinkler.normalized_similarity)
    F["nc_ratio"] = pw(ac, bc, fuzz.ratio); F["nc_tset"] = pw(ac, bc, fuzz.token_set_ratio)
    F["nc_jw"] = pw(ac, bc, JaroWinkler.normalized_similarity); F["nc_lev"] = pw(ac, bc, Levenshtein.normalized_similarity)
    nsa, nsb = [x.replace(" ", "") for x in ac], [x.replace(" ", "") for x in bc]
    F["nsp_ratio"] = pw(nsa, nsb, fuzz.ratio); F["nsp_partial"] = pw(nsa, nsb, fuzz.partial_ratio)
    pa, pb = L(A, "name_ph", iq), L(B, "name_ph", it)   # phonetic skeletons: robust to transliteration
    F["nph_ratio"] = pw(pa, pb, fuzz.ratio); F["nph_tset"] = pw(pa, pb, fuzz.token_set_ratio)
    F["indic_any"] = (A.is_indic.values[iq] | B.is_indic.values[it]).astype(np.int8)
    alt_a, alt_b = L(A, "name_alt", iq), L(B, "name_alt", it)
    has_alt = np.array([bool(x or y) for x, y in zip(alt_a, alt_b)])
    F["nalt_tset"] = F["n_tset"].values
    if has_alt.any():
        ix = np.flatnonzero(has_alt)
        xa = [alt_a[i] or an[i] for i in ix]; xb = [alt_b[i] or bn[i] for i in ix]
        F.loc[ix, "nalt_tset"] = np.maximum(F["n_tset"].values[ix], pw(xa, xb, fuzz.token_set_ratio))
    F["a_tset"] = pw(aa, ba, fuzz.token_set_ratio); F["a_partial"] = pw(aa, ba, fuzz.partial_ratio)
    F["a_ratio"] = pw(aa, ba, fuzz.ratio)
    # TF-IDF cosines + token-set features (transform unique records once, then gather rows)
    tA = {"name_core": A.name_core.astype(str).tolist(), "name_n": A.name_n.astype(str).tolist(),
          "addr_n": A.addr_n.astype(str).tolist(), "full": full_text(A).tolist()}
    tB = {"name_core": B.name_core.astype(str).tolist(), "name_n": B.name_n.astype(str).tolist(),
          "addr_n": B.addr_n.astype(str).tolist(), "full": full_text(B).tolist()}
    for k in FS.sp:
        F[f"cos_{k}"] = rowdot(FS.vec(k, tA[k])[iq], FS.vec(k, tB[k])[it])
    for k, col, w in [("name", "name_core", FS.idf["name"]), ("addr", "addr_n", FS.idf["addr"]), ("num", "addr_n", None)]:
        Xa, Xb = FS.bin[k].transform(tA[col])[iq], par_map(FS.bin[k].transform, tB[col], min_chunk=100_000)[it]
        F[f"{k}_jacc"] = set_feats(Xa, Xb)
        if w is not None: F[f"{k}_idf"] = set_feats(Xa, Xb, w)
    fa, fb = [x.split(" ", 1)[0] for x in ac], [x.split(" ", 1)[0] for x in bc]
    F["first_tok_eq"] = np.array([x == y and x != "" for x, y in zip(fa, fb)], np.int8)
    F["core_eq"] = np.array([x == y for x, y in zip(ac, bc)], np.int8)
    acr_a, acr_b = L(A, "acronym", iq), L(B, "acronym", it)
    F["acr_match"] = np.array([(x != "" and (x == nb or x == y)) or (y != "" and y == na)
                               for x, y, na, nb in zip(acr_a, acr_b, nsa, nsb)], np.int8)
    F["web_any"] = (A.is_web.values[iq] | B.is_web.values[it]).astype(np.int8)
    F["postal_agree"] = agree(A.postal.astype(str).values[iq], B.postal.astype(str).values[it])
    F["fnum_agree"] = agree(A.first_num.astype(str).values[iq], B.first_num.astype(str).values[it])
    na, nb = L(A, "nums", iq), L(B, "nums", it)
    F["nums_seq"] = seq_agree(na, nb)
    F["num_rel"] = np.fromiter((num_rel(x, y) for x, y in zip(na, nb)), np.int8, len(na))
    if "s1_same_name" in A:   # how many records could claim this name (empty-address copies, twins)
        F["q_s1_same_name"] = A.s1_same_name.values[iq]; F["q_t_same_name"] = A.t_same_name.values[iq]
        F["t_s1_same_name"] = B.s1_same_name.values[it]; F["t_t_same_name"] = B.t_same_name.values[it]
    if "nn_cos" in A:   # the S1's closest look-alike: is this copy closer to it than to the S1?
        nnc, nna, nnn = L(A, "nn_name_core", iq), L(A, "nn_addr_n", iq), L(A, "nn_nums", iq)
        F["nn_cos"] = A.nn_cos.values[iq]
        F["nn_nc_tset"] = pw(nnc, bc, fuzz.token_set_ratio); F["nn_a_tset"] = pw(nna, ba, fuzz.token_set_ratio)
        F["nn_a_ratio"] = pw(nna, ba, fuzz.ratio)
        F["nn_nums_seq"] = seq_agree(nnn, nb)
        F["nn_num_rel"] = np.fromiter((num_rel(x, y) for x, y in zip(nnn, nb)), np.int8, len(nnn))
        F["nc_margin"] = F.nc_tset.values - F.nn_nc_tset.values
        F["a_margin"] = F.a_tset.values - F.nn_a_tset.values
        F["num_margin"] = (F.num_rel.values - F.nn_num_rel.values).astype(np.int8)
    F["a_empty"] = np.array([(x == "") or (y == "") for x, y in zip(aa, ba)], np.int8)
    F["len_diff"] = np.abs(np.array([len(x) - len(y) for x, y in zip(ac, bc)], np.float32))
    F["ntok_diff"] = np.abs(np.array([x.count(" ") - y.count(" ") for x, y in zip(ac, bc)], np.float32))
    F["is_s3"] = (B.src.values[it] == 3).astype(np.int8)
    for col in ["cos_full", "cos_name_core", "nc_tset", "a_tset"] + (["dl_cos"] if "dl_cos" in F else []):   # context: rank + gap inside each S1's list
        g = F.groupby("q")[col]
        F[f"{col}_rk"] = g.rank(ascending=False, method="min").astype(np.float32)
        F[f"{col}_gap"] = (g.transform("max") - F[col]).astype(np.float32)
    F["n_cands"] = F.groupby("q").t.transform("size").astype(np.int16)
    return F

_FEAT = {}
def _feat_block(ab):
    global N_PROC, PW_WORKERS
    N_PROC, PW_WORKERS = 1, 1                      # this is a worker: no nested processes / threads
    a, b = ab
    return make_features(_FEAT["c"].iloc[a:b].reset_index(drop=True), _FEAT["Q"], _FEAT["T"], _FEAT["FS"])

def make_features_fast(c, Q, T, FS, min_rows=200_000, timeout=3600):
    """make_features on all CPU cores. The candidate frame (sorted by q) is cut into blocks of complete S1 groups;
    every feature looks at one pair or at one S1's own candidates, so the result is identical to make_features."""
    qv = c.q.values
    k = min(N_PROC, len(c) // min_rows)
    if k < 2 or (np.diff(qv) < 0).any(): return make_features(c, Q, T, FS)
    cuts = np.searchsorted(qv, qv[np.linspace(0, len(c) - 1, k + 1).astype(int)[1:-1]], side="left")
    b = np.unique(np.r_[0, cuts, len(c)])
    _FEAT.update(c=c, Q=Q, T=T, FS=FS)
    try:
        with mp.get_context("fork").Pool(len(b) - 1) as pool:
            parts = pool.map_async(_feat_block, list(zip(b[:-1], b[1:]))).get(timeout)
    except Exception as e:
        print(f"  (parallel features failed, running in one process: {repr(e)[:150]})")
        return make_features(c, Q, T, FS)
    finally:
        _FEAT.clear()
    return pd.concat(parts, ignore_index=True)

NON_FEATS = {"q", "t", "label", "p"}

## 8. Build the TRAIN candidate set + features (chunked) and check blocking recall

In [ ]:
tic("feature spaces"); FS_tr = FeatSpaces(T_tr)
q_of = pd.Index(Q_tr.eid.astype(str)); t_of = pd.Index(T_tr.eid.astype(str))
tp_q = np.concatenate([np.full(len(v), q_of.get_loc(k)) for k, v in gt.items() if v]).astype(np.int64)
tp_t = t_of.get_indexer([x for k, v in gt.items() if v for x in v]).astype(np.int64)
true_keys = np.unique(tp_q * len(T_tr) + tp_t)
gt_counts = pd.Series({k: len(v) for k, v in gt.items()}).reindex(Q_tr.eid.astype(str)).fillna(0).astype(int)
gt_counts.index = np.arange(len(Q_tr))

parts, n_cand = [], 0
for a in range(0, len(Q_tr), S1_CHUNK):
    qdf = Q_tr.iloc[a:a + S1_CHUNK]
    c = BLK_tr.candidates(qdf); c["q"] += a
    n_cand += len(c)
    F = make_features_fast(c, Q_tr, T_tr, FS_tr)
    F["label"] = np.isin(F.q.values * len(T_tr) + F.t.values, true_keys).astype(np.int8)
    for col in F.columns:
        if F[col].dtype == np.float64: F[col] = F[col].astype(np.float32)
    parts.append(F); del c; gc.collect()
    tic(f"  S1 {min(a + S1_CHUNK, len(Q_tr)):,}/{len(Q_tr):,}  pairs so far {n_cand:,}")
F_tr = pd.concat(parts, ignore_index=True); del parts; gc.collect()
FEATS = [c for c in F_tr.columns if c not in NON_FEATS]

hit = int(F_tr.label.sum())
print(f"\nPAIR RECALL: {hit / len(true_keys):.4f}  ({hit:,}/{len(true_keys):,})   mean cands/S1: {len(F_tr)/len(Q_tr):.1f}")
for b in ["ann_name", "ann_full", "ann_addr", "ann_dl", "rev", "rare", "akey", "nakey", "phon", "hop"]:
    if f"{b}_rank" not in F_tr: continue
    m = F_tr[f"{b}_rank"] < 99
    print(f"   {b:9s} alone recall {F_tr.label[m].sum()/len(true_keys):.4f}   pairs {m.sum():,}")
hits_per_q = F_tr.groupby("q").label.sum().reindex(range(len(Q_tr)), fill_value=0).values
ntrue = gt_counts.values
with np.errstate(divide="ignore", invalid="ignore"):
    orc = np.where(ntrue == 0, 1.0, np.where(hits_per_q > 0, 1.25 * (hits_per_q / ntrue) / (0.25 + hits_per_q / ntrue), 0))
print(f"oracle macro F0.5 (ceiling): {orc.mean():.4f}  | features: {len(FEATS)}  | RAM for F_tr: {F_tr.memory_usage().sum()/1e9:.2f} GB")

## 9. Metric + decision logic (thresholds tuned directly on macro F0.5)

In [ ]:
def decide(df, p, t_match, t_empty, alpha, one_to_one=True, margin=0.0):
    """df pre-sorted by p descending. Keep p ≥ t_match; one-to-one: each S2/S3 record only for its best S1;
    relative cut p ≥ alpha·best; singleton rule: best < t_empty → predict nothing. Unseen countries get +margin."""
    u = df["unseen"].values if "unseen" in df else 0
    d = df[df[p].values >= t_match + margin * u]
    if one_to_one: d = d.drop_duplicates("t")
    mx = d.groupby("q")[p].transform("max")
    uu = d["unseen"].values if "unseen" in d else 0
    return d[(d[p] >= alpha * mx) & (mx >= t_empty + margin * uu)]

def fast_score(sel, gt_counts):
    """Vectorized macro F0.5 over ALL S1 in gt_counts (index = q)."""
    agg = sel.groupby("q").label.agg(["sum", "size"])
    tp = agg["sum"].reindex(gt_counts.index, fill_value=0).values
    npred = agg["size"].reindex(gt_counts.index, fill_value=0).values
    ntrue = gt_counts.values
    with np.errstate(divide="ignore", invalid="ignore"):
        pr, rc = tp / np.maximum(npred, 1), tp / np.maximum(ntrue, 1)
        f = np.where(tp > 0, 1.25 * pr * rc / (0.25 * pr + rc), 0.0)
    return float(np.where((ntrue == 0) & (npred == 0), 1.0, f).mean())

def tune(df, p, gt_counts):
    df = df[["q", "t", p, "label"]].sort_values(p, ascending=False)
    best = (-1, None)
    for oto in (True, False):
        for tm in np.arange(0.05, 0.96, 0.025):
            s = fast_score(decide(df, p, tm, tm, 0.0, oto), gt_counts)
            if s > best[0]: best = (s, dict(t_match=float(tm), t_empty=float(tm), alpha=0.0, one_to_one=oto))
    cfg = best[1]
    for te in np.arange(cfg["t_match"], 0.99, 0.025):
        for al in (0.0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8):
            s = fast_score(decide(df, p, cfg["t_match"], te, al, cfg["one_to_one"]), gt_counts)
            if s > best[0]: best = (s, dict(cfg, t_empty=float(te), alpha=al))
    cfg = best[1]
    for tm in np.arange(cfg["t_match"] - 0.015, cfg["t_match"] + 0.016, 0.005):   # fine pass around the best point
        for te in np.arange(max(tm, cfg["t_empty"] - 0.015), cfg["t_empty"] + 0.016, 0.005):
            s = fast_score(decide(df, p, tm, te, cfg["alpha"], cfg["one_to_one"]), gt_counts)
            if s > best[0]: best = (s, dict(cfg, t_match=float(tm), t_empty=float(te)))
    return best

## 10. XGBoost on GPU (or LightGBM on CPU) with GroupKFold (groups = S1) → out-of-fold predictions → tuned thresholds

In [ ]:
PARAMS = dict(objective="binary", metric="auc", learning_rate=0.08, num_leaves=127, min_child_samples=100,
              feature_fraction=0.8, bagging_fraction=0.7, bagging_freq=1, lambda_l2=1.0, max_bin=127,
              verbose=-1, seed=SEED, n_jobs=-1)
# XGBoost counterpart of PARAMS (leaf-wise trees with the same leaf count, sampling and learning rate)
XGB_PARAMS = dict(objective="binary:logistic", eval_metric="auc", device=KNN_DEVICE, tree_method="hist",
                  grow_policy="lossguide", max_depth=0, max_leaves=127, learning_rate=0.08, subsample=0.7,
                  colsample_bytree=0.8, reg_lambda=1.0, max_bin=128, seed=SEED, verbosity=0)

class XGBModel:
    """XGBoost Booster with the LightGBM methods used below (predict / save_model / feature_importance)."""
    def __init__(self, booster, feats):
        self.b, self.feats = booster, list(feats)
        self.best_iteration = booster.best_iteration + 1
        self.best_score = {"valid_0": {"auc": booster.best_score}}
    def predict(self, X, num_iteration=None, num_threads=None):
        it = (0, num_iteration or self.best_iteration)
        A = np.ascontiguousarray(X.to_numpy(np.float32) if hasattr(X, "to_numpy") else X, dtype=np.float32)
        try:
            import cupy as cp   # data straight to the GPU → no device-mismatch copy inside XGBoost
            out = cp.asnumpy(self.b.inplace_predict(cp.asarray(A), iteration_range=it, validate_features=False))
            XGBModel.path = "GPU (CuPy → XGBoost)"; return out
        except Exception:
            XGBModel.path = "GPU (DMatrix copy)" if KNN_DEVICE == "cuda" else "CPU"
            return self.b.predict(xgb.DMatrix(A), iteration_range=it, validate_features=False)
    def save_model(self, path):
        self.b.save_model(os.path.splitext(path.replace("lgb_", "xgb_"))[0] + ".json")
    def feature_importance(self, kind="gain"):
        s = self.b.get_score(importance_type="total_gain")
        return np.array([s.get(f, 0.0) for f in self.feats], np.float64)

def fit_fold(Xtr, ytr, Xva, yva):
    """GPU XGBoost when available (falls back to LightGBM for good if it fails), else LightGBM on CPU."""
    global MODEL
    if MODEL == "xgb":
        try:
            dtr = xgb.QuantileDMatrix(Xtr, ytr, max_bin=XGB_PARAMS["max_bin"])
            dva = xgb.QuantileDMatrix(Xva, yva, ref=dtr, max_bin=XGB_PARAMS["max_bin"])
            b = xgb.train(XGB_PARAMS, dtr, 3000, evals=[(dva, "valid_0")], early_stopping_rounds=100,
                          verbose_eval=False)
            del dtr, dva
            return XGBModel(b, Xtr.columns)
        except Exception as e:
            MODEL = "lgb"; print("  XGBoost GPU failed, using LightGBM on CPU from now on:", repr(e)[:200])
    return lgb.train(PARAMS, lgb.Dataset(Xtr, ytr, free_raw_data=True), 3000, valid_sets=[lgb.Dataset(Xva, yva)],
                     callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])

try: gkf = GroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
except TypeError: gkf = GroupKFold(n_splits=N_FOLDS)
if MODEL == "xgb" and GPU_KNN and "BLK_tr" in globals() and torch.cuda.get_device_properties(0).total_memory < 40e9:
    # small GPU: park the train kNN vectors in CPU RAM while XGBoost trains (10a moves them back when it searches),
    # but only with RAM to spare: out of RAM kills the runtime, out of GPU memory only falls back to LightGBM
    _need = sum(ix.X.numel() * 2 for ix in BLK_tr.idx.values() if isinstance(ix, TorchFlatIP))
    try: import psutil; _free = psutil.virtual_memory().available
    except ImportError: _free = 0
    if _free > _need + 1.5 * F_tr.memory_usage().sum():
        BLK_tr.offload(); print(f"train kNN vectors ({_need / 1e9:.1f} GB) parked in CPU RAM for model training | {gpu_mem()}")
    else:
        print(f"train kNN vectors stay on the GPU (RAM free {_free / 1e9:.0f} GB is too little to park {_need / 1e9:.1f} GB)")
oof, models = np.zeros(len(F_tr), np.float32), []
y_all = F_tr.label.values
_fcols = [F_tr.columns.get_loc(c) for c in FEATS]              # fold slices straight from F_tr (no full copy in RAM)
for fold, (tr, va) in enumerate(gkf.split(F_tr.q.values, y_all, groups=F_tr.q.values)):
    Xva = F_tr.iloc[va, _fcols]
    m = fit_fold(F_tr.iloc[tr, _fcols], y_all[tr], Xva, y_all[va])
    oof[va] = m.predict(Xva, num_iteration=m.best_iteration)
    models.append(m); del Xva; tic(f"fold {fold}: iters={m.best_iteration} auc={m.best_score['valid_0']['auc']:.5f}")
    gc.collect()
F_tr["p"] = oof
best_score, CFG = tune(F_tr[F_tr.p >= KEEP_P], "p", gt_counts)   # same result, much faster
print(f"\nOOF macro F0.5 = {best_score:.4f}   with {CFG}   | model: {MODEL}  (local v5 LightGBM: 0.9507)")
imp = pd.Series(models[0].feature_importance("gain"), index=FEATS).sort_values(ascending=False)
imp.head(20)

## 11. Leave-one-country-out (proxy for unseen France)

In [ ]:
if RUN_LOCO:
    ck = Q_tr.country_key.astype(str).values[F_tr.q.values]
    n_iter = int(np.mean([m.best_iteration for m in models])) or 300
    for c in sorted(set(ck)):
        trm = ck != c
        if trm.all() or (~trm).all(): continue
        m = lgb.train(PARAMS, lgb.Dataset(F_tr.loc[trm, FEATS], F_tr.label.values[trm]), n_iter)
        d = F_tr.loc[~trm, ["q", "t", "label"]].copy(); d["pl"] = m.predict(F_tr.loc[~trm, FEATS])
        qs = np.flatnonzero(Q_tr.country_key.astype(str).values == c)
        s = fast_score(decide(d.sort_values("pl", ascending=False), "pl", **CFG), gt_counts.loc[qs])
        print(f"LOCO held-out {c}: macro F0.5 = {s:.4f}"); del m, d; gc.collect()

### 10a. Cross-encoder re-ranker (fine-tuned MiniLM, Apache-2.0)
A small pretrained transformer (`cross-encoder/ms-marco-MiniLM-L-6-v2`, 22M parameters, Apache-2.0) is fine-tuned on
~4M candidate pairs of S1 records that are **not** in the GBDT training sample. It reads both records as text
("name ; address") and learns the noise directly: typos, made-up names, generic replacements, number noise, twins.
Its score is a stage-2 feature (10b), used only if the out-of-fold F0.5 improves. GPU runtimes only.

In [ ]:
if USE_CE:
    try:
        from transformers import AutoTokenizer, AutoModelForSequenceClassification, get_linear_schedule_with_warmup
        from sklearn.metrics import roc_auc_score
        torch.manual_seed(SEED)
        tic(f"cross-encoder: loading {CE_MODEL}")
        ce_tok = AutoTokenizer.from_pretrained(CE_MODEL)
        ce_model = AutoModelForSequenceClassification.from_pretrained(CE_MODEL, num_labels=1).to(KNN_DEVICE)

        def rec_text(df):
            """What the cross-encoder reads for one record: normalized name ; normalized address."""
            return (df["name_n"].astype(str) + " ; " + df["addr_n"].astype(str)).tolist()

        @torch.no_grad()
        def ce_score(a_txt, b_txt, bs=2048):
            """Match probability of text pairs; pairs are sorted by length so batches carry little padding."""
            ce_model.eval()
            out = np.empty(len(a_txt), np.float32)
            order = np.argsort(np.fromiter((len(x) + len(y) for x, y in zip(a_txt, b_txt)), np.int64, len(a_txt)),
                               kind="stable")
            for s in range(0, len(order), bs):
                ix = order[s:s + bs]
                enc = ce_tok([a_txt[i] for i in ix], [b_txt[i] for i in ix], truncation=True, max_length=CE_MAXLEN,
                             padding=True, return_tensors="pt").to(KNN_DEVICE)
                with torch.autocast(device_type=KNN_DEVICE, dtype=torch.bfloat16, enabled=KNN_DEVICE == "cuda"):
                    out[ix] = torch.sigmoid(ce_model(**enc).logits.float().squeeze(-1)).cpu().numpy()
            return out

        def add_ce_feats(D, Q, T, block=2_000_000):
            """Cross-encoder score of every kept pair + its context among the same S1's candidates."""
            qtxt = np.array(rec_text(Q), dtype=object)
            ce = np.empty(len(D), np.float32)
            for a in range(0, len(D), block):
                sl = slice(a, a + block)
                ce[sl] = ce_score(qtxt[D.q.values[sl]].tolist(), rec_text(T.iloc[D.t.values[sl]]))
            D["ce"] = ce
            g = D.groupby("q").ce
            D["ce_max"] = g.transform("max").astype(np.float32)
            D["ce_gap"] = (D.ce_max - D.ce).astype(np.float32)
            D["ce_rank"] = g.rank(ascending=False, method="first").astype(np.float32)
            D["ce_n_hi"] = (D.ce >= 0.5).groupby(D.q).transform("sum").astype(np.float32)
            return D

        # ---- training pairs: S1 records NOT in the GBDT training sample (so the stage-2 feature stays honest) ----
        _in = set(Q_tr.eid.astype(str))
        _pool = S1ALL_tr[~S1ALL_tr.eid.astype(str).isin(_in)]
        _pool = _pool.sample(min(CE_POOL_S1, len(_pool)), random_state=SEED).reset_index(drop=True)
        _t_of = pd.Index(T_tr.eid.astype(str))
        _pq, _pt = [], []
        for i, e in enumerate(_pool.eid.astype(str)):
            for x in gt_dict.get(e, ()): _pq.append(i); _pt.append(x)
        _pos = pd.DataFrame({"q": np.array(_pq, np.int64), "t": _t_of.get_indexer(_pt).astype(np.int64)})
        _pos = _pos[_pos.t >= 0].drop_duplicates()
        tic(f"cross-encoder: candidates of {len(_pool):,} pool S1")
        _neg = []
        for a in range(0, len(_pool), 50_000):
            c = BLK_tr.candidates(_pool.iloc[a:a + 50_000]); c["q"] += a
            c["h"] = c.ann_full_score + c.ann_name_score        # blocker similarity → hardest negatives first
            _neg.append(c[["q", "t", "h"]]); del c
        _neg = pd.concat(_neg, ignore_index=True)
        _is_pos = _neg.merge(_pos.assign(y=1), on=["q", "t"], how="left").y.notna().values
        _found = _is_pos.sum() / max(len(_pos), 1)
        _neg = _neg[~_is_pos].sort_values(["q", "h"], ascending=[True, False])
        _hard = _neg.groupby("q").head(CE_NEG_PER_S1)
        _rest = _neg.drop(_hard.index).sample(frac=1, random_state=SEED).groupby("q").head(CE_RAND_NEG_PER_S1)
        _P = pd.concat([_pos.assign(y=1), _hard[["q", "t"]].assign(y=0), _rest[["q", "t"]].assign(y=0)],
                       ignore_index=True)
        _qa = np.array(rec_text(_pool), dtype=object)[_P.q.values]
        _tb = np.array(rec_text(T_tr.iloc[_P.t.values]), dtype=object)
        _y = _P.y.values.astype(np.float32)
        _va = (_P.q.values % 20) == 0                        # 5% of the pool S1 held out for validation
        _tr_i, _va_i = np.flatnonzero(~_va), np.flatnonzero(_va)
        print(f"  {len(_P):,} pairs ({_y.mean()*100:.1f}% matches) | the blocker finds {_found*100:.1f}% of the pool's "
              f"true pairs; the rest are still training positives")

        # ---- fine-tune ----
        steps = CE_EPOCHS * int(np.ceil(len(_tr_i) / CE_BATCH))
        opt = torch.optim.AdamW(ce_model.parameters(), lr=CE_LR, weight_decay=0.01)
        sch = get_linear_schedule_with_warmup(opt, int(0.05 * steps), steps)
        lossf = torch.nn.BCEWithLogitsLoss()
        tic(f"cross-encoder: fine-tuning, {steps:,} steps of {CE_BATCH}")
        ce_model.train(); _rs = np.random.default_rng(SEED); step = 0; _run = []
        for ep in range(CE_EPOCHS):
            perm = _rs.permutation(_tr_i)
            for s in range(0, len(perm), CE_BATCH):
                ix = perm[s:s + CE_BATCH]
                enc = ce_tok(_qa[ix].tolist(), _tb[ix].tolist(), truncation=True, max_length=CE_MAXLEN,
                             padding=True, return_tensors="pt").to(KNN_DEVICE)
                with torch.autocast(device_type=KNN_DEVICE, dtype=torch.bfloat16, enabled=KNN_DEVICE == "cuda"):
                    logit = ce_model(**enc).logits.float().squeeze(-1)
                loss = lossf(logit, torch.from_numpy(_y[ix]).to(KNN_DEVICE))
                loss.backward(); torch.nn.utils.clip_grad_norm_(ce_model.parameters(), 1.0)
                opt.step(); sch.step(); opt.zero_grad(set_to_none=True); step += 1; _run.append(loss.item())
                if step % 2000 == 0 or step == steps:
                    tic(f"  step {step:,}/{steps:,}  loss {np.mean(_run[-500:]):.4f} | {gpu_mem()}")
        t0 = time.time(); _pv = ce_score(_qa[_va_i].tolist(), _tb[_va_i].tolist()); sec = time.time() - t0
        print(f"  validation (held-out pool S1): AUC {roc_auc_score(_y[_va_i], _pv):.5f} | accuracy@0.5 "
              f"{((_pv >= 0.5) == (_y[_va_i] == 1)).mean():.4f} | scoring {len(_va_i) / max(sec, 1e-9):,.0f} pairs/s")
        del _pool, _pos, _neg, _hard, _rest, _P, _qa, _tb, _y, _pv, opt, sch
    except Exception as e:
        USE_CE = False
        print("cross-encoder failed, continuing without it:", repr(e)[:300])
    for _v in ["gt_dict", "S1ALL_tr"]: globals().pop(_v, None)
    gc.collect()
    if GPU_KNN: torch.cuda.empty_cache()
else:
    print("cross-encoder off (needs a GPU runtime with transformers)")

## 10b. Stage 2 + per-country thresholds
A second small model re-scores each pair using the other candidates of the same S1 (best score, gap to it,
how many are high, per source). Thresholds are then tuned per training country; France (unseen) uses the global
ones plus `UNSEEN_COUNTRY_MARGIN`. Each part is only switched on if it beats stage 1 on out-of-fold F0.5.

In [ ]:
S2_PARAMS = dict(objective="binary", metric="auc", learning_rate=0.05, num_leaves=63 if USE_CE else 31, min_child_samples=200,
                 feature_fraction=0.9, bagging_fraction=0.8, bagging_freq=1, lambda_l2=1.0,
                 verbose=-1, seed=SEED, n_jobs=-1)
S2_FEATS = ["p", "p_rank", "p_max", "p_gap", "p_rel", "p_sum", "n_kept", "n_hi", "p_2nd",
            "p_rank_src", "p_max_src", "n_hi_src", "is_s3"]
S2_BASE = [c for c in ("cos_full", "nc_tset", "n_tset", "a_tset", "a_empty", "num_rel", "nums_seq", "fnum_agree",
                       "nn_cos", "nc_margin", "a_margin", "q_s1_same_name", "t_s1_same_name",
                       "dl_cos", "dl_t_gap", "dl_nn_margin", "rev_rank", "rev_nf")
          if c in FEATS]                                                    # stage-1 features stage 2 also sees
CE_FEATS = ["ce", "ce_max", "ce_gap", "ce_rank", "ce_n_hi"] if USE_CE else []
S2_ALL = S2_FEATS + S2_BASE + CE_FEATS

def ctx_feats(d):
    """d has q, t, p, is_s3 (filtered to p >= KEEP_P). Adds context of the other candidates of the same S1."""
    d = d.sort_values(["q", "p"], ascending=[True, False]).reset_index(drop=True)
    gq = d.groupby("q").p
    d["p_rank"] = (d.groupby("q").cumcount() + 1).astype(np.float32)
    d["p_max"]  = gq.transform("max").astype(np.float32)
    d["p_gap"]  = (d.p_max - d.p).astype(np.float32)
    d["p_rel"]  = (d.p / d.p_max.clip(lower=1e-6)).astype(np.float32)
    d["p_sum"]  = gq.transform("sum").astype(np.float32)
    d["n_kept"] = gq.transform("size").astype(np.float32)
    d["n_hi"]   = (d.p >= 0.5).groupby(d.q).transform("sum").astype(np.float32)
    d["p_2nd"]  = d.q.map(d.loc[d.p_rank == 2].set_index("q").p).fillna(0).astype(np.float32)
    d["p_rank_src"] = (d.groupby(["q", "is_s3"]).cumcount() + 1).astype(np.float32)
    d["p_max_src"]  = d.groupby(["q", "is_s3"]).p.transform("max").astype(np.float32)
    d["n_hi_src"]   = (d.p >= 0.5).groupby([d.q, d.is_s3]).transform("sum").astype(np.float32)
    return d

def tune_by_country(D, pcol):
    qc = Q_tr.country_key.astype(str).values
    cfgs, total = {}, 0.0
    for c in np.unique(qc):
        qs = np.flatnonzero(qc == c)
        s, cfg = tune(D[np.isin(D.q.values, qs)], pcol, gt_counts.loc[qs])
        cfgs[c] = cfg; total += s * len(qs)
        print(f"   {c}: F0.5 {s:.4f}  {cfg}")
    return total / len(qc), cfgs

USE_STAGE2, USE_PER_COUNTRY, models2, CFG_C = False, False, [], {}
P_COL, CFG_GLOBAL, FINAL_OOF = "p", CFG, best_score
if TRY_STAGE2:
    tic("stage 2: context features")
    D_tr = ctx_feats(F_tr.loc[F_tr.p >= KEEP_P, ["q", "t", "p", "is_s3", "label"] + S2_BASE])
    if USE_CE:
        tic(f"stage 2: cross-encoder on {len(D_tr):,} kept train pairs"); D_tr = add_ce_feats(D_tr, Q_tr, T_tr)
        print(f"  AUC on these pairs: cross-encoder {roc_auc_score(D_tr.label, D_tr.ce):.5f} | "
              f"stage-1 GBDT {roc_auc_score(D_tr.label, D_tr.p):.5f}")
    X2, y2 = D_tr[S2_ALL], D_tr.label.values
    oof2 = np.zeros(len(D_tr), np.float32)
    for fold, (tr, va) in enumerate(gkf.split(X2, y2, groups=D_tr.q.values)):
        m = lgb.train(S2_PARAMS, lgb.Dataset(X2.iloc[tr], y2[tr]), 2000,
                      valid_sets=[lgb.Dataset(X2.iloc[va], y2[va])],
                      callbacks=[lgb.early_stopping(100, verbose=False), lgb.log_evaluation(0)])
        oof2[va] = m.predict(X2.iloc[va], num_iteration=m.best_iteration); models2.append(m)
        tic(f"  stage-2 fold {fold}: iters={m.best_iteration}")
    D_tr["p2"] = oof2
    score2, CFG2 = tune(D_tr, "p2", gt_counts)
    print(f"\nstage 1 OOF F0.5 = {best_score:.4f} | stage 2 OOF F0.5 = {score2:.4f}  {CFG2}")
    USE_STAGE2 = score2 > best_score + 0.001
    if USE_STAGE2: P_COL, CFG_GLOBAL, FINAL_OOF = "p2", CFG2, score2
    print("per-country thresholds on", P_COL)
    score_c, CFG_C = tune_by_country(D_tr, P_COL)
    USE_PER_COUNTRY = score_c > FINAL_OOF + 0.0005
    if USE_PER_COUNTRY: FINAL_OOF = score_c
    del X2, y2, oof2; gc.collect()
print(f"\nUSE_STAGE2={USE_STAGE2}  USE_PER_COUNTRY={USE_PER_COUNTRY}  → final OOF F0.5 = {FINAL_OOF:.4f}")

### 10c. Error analysis — where is the F0.5 lost?
Uses the final out-of-fold decision (same model, thresholds, stage 2 / per-country choice as the submission) and splits
the lost F0.5 by country and cause: singletons given a match, pairs blocking never found, pairs the model rejected,
extra wrong matches. Then shows real examples. Must run here: section 12 frees the training data.

In [ ]:
if RUN_ERROR_ANALYSIS:
    _src = D_tr if (P_COL != "p" and "D_tr" in globals()) else F_tr
    _d = _src.loc[_src[P_COL] >= KEEP_P, ["q", "t", P_COL, "label"]].sort_values(P_COL, ascending=False)
    _qc = Q_tr.country_key.astype(str).values
    _sel = pd.concat([decide(_d[_qc[_d.q.values] == c], P_COL,
                             **(CFG_C.get(c, CFG_GLOBAL) if USE_PER_COUNTRY else CFG_GLOBAL))
                      for c in np.unique(_qc[_d.q.values])], ignore_index=True)
    _agg = _sel.groupby("q").label.agg(["sum", "size"])
    tp = _agg["sum"].reindex(gt_counts.index, fill_value=0).values
    npred = _agg["size"].reindex(gt_counts.index, fill_value=0).values
    ntrue = gt_counts.values
    _cq = _qc[gt_counts.index]   # kept for section 13: test vs out-of-fold predicted matches per S1
    OOF_STATS = dict(pred=pd.Series(npred).groupby(_cq).mean().to_dict(), true=pd.Series(ntrue).groupby(_cq).mean().to_dict(),
                     none=(pd.Series(npred == 0).groupby(_cq).mean() * 100).to_dict())
    found = F_tr.groupby("q").label.sum().reindex(gt_counts.index, fill_value=0).values   # true pairs among candidates
    with np.errstate(divide="ignore", invalid="ignore"):
        pr, rc = tp / np.maximum(npred, 1), tp / np.maximum(ntrue, 1)
        f = np.where(tp > 0, 1.25 * pr * rc / (0.25 * pr + rc), 0.0)
    f = np.where((ntrue == 0) & (npred == 0), 1.0, f)
    cat = np.select(
        [(ntrue == 0) & (npred == 0), (ntrue == 0) & (npred > 0),
         (ntrue > 0) & (npred == 0) & (found == 0), (ntrue > 0) & (npred == 0),
         (tp == ntrue) & (npred == tp), (tp == ntrue) & (npred > tp),
         (tp < ntrue) & (npred == tp) & (found > tp), (tp < ntrue) & (npred == tp)],
        ["singleton OK", "singleton but matched", "nothing predicted, blocking missed all",
         "nothing predicted, model rejected", "perfect", "all found + extra wrong",
         "missed some, model rejected", "missed some, blocking missed"], "missed some + extra wrong")
    R = pd.DataFrame({"country": _qc[gt_counts.index], "cat": cat, "loss": 1 - f})
    print(f"macro F0.5 {f.mean():.4f} (should equal the final OOF {FINAL_OOF:.4f}) | total loss {1 - f.mean():.4f}")
    _t = R.groupby(["country", "cat"]).agg(n=("loss", "size"), loss=("loss", "sum"))
    _t["F05_points_lost"] = (_t.loss / len(R) * 100).round(3); _t["share_%"] = (_t.loss / R.loss.sum() * 100).round(1)
    print(_t.drop(columns="loss").sort_values("share_%", ascending=False).to_string())

    # pair-level: false matches / missed true pairs, with the main features and how common the S1 name is
    _m = F_tr[["q", "t", "label", "cos_full", "nc_tset", "a_tset", "fnum_agree"]]
    _m = _m.merge(_src[["q", "t", P_COL]], on=["q", "t"], how="left").fillna({P_COL: 0})
    _m = _m.merge(_sel[["q", "t"]].assign(s=1), on=["q", "t"], how="left")
    fp = _m[(_m.s == 1) & (_m.label == 0)].sort_values(P_COL, ascending=False)
    fn = _m[(_m.s != 1) & (_m.label == 1)]
    _freq = Q_tr.name_core.astype(str).map(T_tr.name_core.astype(str).value_counts()).fillna(0).values
    _common = lambda df: (_freq[df.q.values] > 12).mean() * 100 if len(df) else 0.0
    print(f"\nfalse matches: {len(fp):,} ({_common(fp):.0f}% have an S1 name shared by >12 S2/S3 records) | "
          f"missed true pairs inside candidates: {len(fn):,} ({_common(fn):.0f}% common names) | "
          f"all S1: {(_freq > 12).mean() * 100:.0f}% common names")
    print("score of missed true pairs (quantiles):", fn[P_COL].quantile([.1, .25, .5, .75, .9]).round(3).to_dict())
    def _show(df, title):
        print(f"\n--- {title}")
        for r in df.itertuples():
            a, b = Q_tr.iloc[r.q], T_tr.iloc[r.t]
            print(f"p={getattr(r, P_COL):.3f} cos={r.cos_full:.2f} n={r.nc_tset:.0f} a={r.a_tset:.0f} num={r.fnum_agree} "
                  f"name_freq={_freq[r.q]:.0f} [{a.country_key}]"
                  f"\n   S1 {a.name_n!s:.45} | {a.addr_n!s:.60}\n   →  {b.name_n!s:.45} | {b.addr_n!s:.60}")
    _show(fp.head(12), "FALSE MATCHES with the highest score")
    _show(fn.sort_values(P_COL, ascending=False).head(10), "MISSED true pairs just under the threshold")
    _show(fn.sample(min(10, len(fn)), random_state=0), "MISSED true pairs, random sample")
    del _src, _d, _sel, _agg, _m, fp, fn, R, _t, _freq; gc.collect()

## 12. Free train memory, save models, load TEST

In [ ]:
from pathlib import Path

# ---- Output folders: every run gets its own folder, so a new run never overwrites an old submission ----
# Colab: the run is written to the fast local disk and finished files are copied to MyDrive/amazon_ml_2026,
# because /content is wiped when the runtime is deleted.
OUT_DIR = "/content/amazon_ml_2026" if IN_COLAB else str(Path.home() / "Downloads" / "amazon_ml_2026")
DRIVE_OUT = ""  # Optional: set to "/content/drive/MyDrive/amazon_ml_2026" if saving to Drive
RUN_DIR = os.path.join(OUT_DIR, "runs", time.strftime("%Y%m%d_%H%M") + f"_oof{FINAL_OOF:.4f}")
os.makedirs(RUN_DIR, exist_ok=True)
print("RUN_DIR:", RUN_DIR)

def save_to_drive(*paths):
    """Colab only: copy finished files to MyDrive/amazon_ml_2026/runs/<this run>/."""
    if not DRIVE_OUT: return
    dst = os.path.join(DRIVE_OUT, "runs", os.path.basename(RUN_DIR)); os.makedirs(dst, exist_ok=True)
    for p in paths: shutil.copy2(p, dst)
    print(f"copied {len(paths)} file(s) to Google Drive → {dst}")

for i, m in enumerate(models):
    m.save_model(os.path.join(RUN_DIR, f"lgb_fold{i}.txt"))
for i, m in enumerate(models2):
    m.save_model(os.path.join(RUN_DIR, f"lgb_stage2_fold{i}.txt"))
with open(os.path.join(RUN_DIR, "config.json"), "w") as f:
    json.dump(dict(CFG=CFG, oof_f05_stage1=best_score, final_oof_f05=FINAL_OOF, FEATS=FEATS,
                   USE_STAGE2=USE_STAGE2, USE_PER_COUNTRY=USE_PER_COUNTRY, P_COL=P_COL,
                   CFG_GLOBAL=CFG_GLOBAL, CFG_C=CFG_C,
                   K=dict(NAME=K_NAME, FULL=K_FULL, RARE=K_RARE, AKEY=K_AKEY, PHON=K_PHON, HOP=K_HOP, ADDR=K_ADDR, NAKEY=K_NAKEY),
                   USE_PHON=USE_PHON, USE_HOP=USE_HOP, HOP_SEED_MIN=HOP_SEED_MIN, SVD_DIM=SVD_DIM,
                   MATCH_TEST_CROWD=MATCH_TEST_CROWD, USE_REV=USE_REV, K_REV=K_REV, REV_FIELDS=list(REV_FIELDS), USE_DL=DL_ENC is not None, K_DL=K_DL,
                   DL_DIM=DL_DIM, NPROBE=NPROBE, TRAIN_S1_FRAC=TRAIN_S1_FRAC, MODEL=MODEL, GPU_KNN=GPU_KNN, USE_CE=USE_CE, CE_MODEL=CE_MODEL, S2_ALL=S2_ALL, margin=UNSEEN_COUNTRY_MARGIN), f, indent=2)

with open(os.path.join(RUN_DIR, "translit_map.json"), "w", encoding="utf-8") as f:
    json.dump(TMAP, f, ensure_ascii=False)
print(f"saved {len(TMAP):,} transliteration mappings (used again for TEST normalization)")
if DL_ENC is not None:
    torch.save(DL_ENC.net.state_dict(), os.path.join(RUN_DIR, "dl_encoder.pt")); print("saved the learned encoder")
save_to_drive(*[os.path.join(RUN_DIR, f) for f in os.listdir(RUN_DIR) if os.path.isfile(os.path.join(RUN_DIR, f))])
if USE_CE:
    _ced = os.path.join(RUN_DIR, "cross_encoder"); ce_model.save_pretrained(_ced); ce_tok.save_pretrained(_ced)
    if DRIVE_OUT:
        shutil.copytree(_ced, os.path.join(DRIVE_OUT, "runs", os.path.basename(RUN_DIR), "cross_encoder"), dirs_exist_ok=True)
    print("saved the fine-tuned cross-encoder →", _ced)

print("before freeing train:", gpu_mem())
for _v in ["F_tr", "D_tr", "X_all", "y_all", "oof", "BLK_tr", "FS_tr", "Q_tr", "T_tr", "tr1", "imp",
           "d", "sel", "agg", "tp", "npred", "ntrue", "found", "f", "cat", "R", "t", "m", "fp", "fn",   # error analysis
           "c", "F", "qdf", "parts", "true_keys", "tp_q", "tp_t", "S1ALL_tr", "tr1_all", "gt_dict"]:
    globals().pop(_v, None)
gc.collect()
if GPU_KNN: torch.cuda.empty_cache()
try:
    import psutil; print(f"after freeing train: {gpu_mem()} | RAM free {psutil.virtual_memory().available/1e9:.0f} GB")
except ImportError: print("after freeing train:", gpu_mem())

tic("loading TEST")
te1, te2, te3 = [read_tsv(os.path.join(TEST_DIR, f"test_source{i}.tsv")) for i in (1, 2, 3)]
S1_TEST_IDS = te1.entity_id.astype(str).tolist()
tic("normalizing TEST"); Q_te, T_te = prep_stage(te1, te2, te3)
del te1, te2, te3; gc.collect()
tic("building TEST blocker"); BLK_te = Blocker(Q_te, T_te)
if USE_REV: tic("reverse kNN (TEST): every S2/S3 record → its closest S1 among all test S1"); BLK_te.build_reverse(Q_te)
if "TB_REF" in globals() and "dl" in BLK_te.t_best:
    # label-free check: test's closest-S1 similarities = a mix of "has an owner" and "orphan" (train shapes) → mix weight
    _tb = BLK_te.t_best["dl"].astype(np.float32); _tc = T_te.country_key.astype(str).to_numpy(dtype=object)
    _d = TB_REF["orph"] - TB_REF["own"]
    print(f"estimated share of orphan S2/S3 records (no S1 record) — train {TB_REF['share'] * 100:.1f}%:")
    for c in np.unique(_tc):
        h = np.histogram(_tb[_tc == c], TB_BINS, density=True)[0]
        w = float(np.clip(np.dot(h - TB_REF["own"], _d) / max(float(np.dot(_d, _d)), 1e-12), 0, 1))
        print(f"   test {c}: {w * 100:.1f}%" + ("" if c in TRAIN_COUNTRIES else "  (country not in train: rough)"))
    del _tb, _tc, _d
if USE_S1_NN: tic("S1 look-alikes (TEST)"); add_s1_neighbours(Q_te, Q_te, BLK_te.emb["full"])
tic("S1 / S2-S3 name counts (TEST)"); add_name_counts(Q_te, T_te, Q_te)
tic("TEST feature spaces"); FS_te = FeatSpaces(T_te); tic(f"ready | {gpu_mem()}")
print(f"blocking used for TEST: NPROBE={NPROBE} K_NAME={K_NAME} K_FULL={K_FULL} USE_PHON={USE_PHON} USE_HOP={USE_HOP} "
      f"reverse kNN={BLK_te.rev_s1 is not None} learned kNN K_DL={K_DL if 'dl' in BLK_te.emb else 0}")

## 13. Chunked test inference
For each chunk of S1: candidates → write them to `candidate_pairs.tsv` → features → average of fold models.
Only pairs with p ≥ `KEEP_P` are kept for the final global decision (one-to-one needs all S1 at once).

In [ ]:
N_THREADS = os.cpu_count()
CKPT = os.path.join(RUN_DIR, "ckpt_test"); os.makedirs(CKPT, exist_ok=True)   # per run: never reuses old chunks
# (If you interrupt this cell, just run it again: finished chunks are skipped. Don't re-run section 12 in between,
#  because that creates a new RUN_DIR.)

t_eid = T_te.eid.astype(str).to_numpy(dtype=object)
unseen_q = (~Q_te.country_key.astype(str).isin(TRAIN_COUNTRIES)).values.astype(np.int8)

def join_lists(df, t_eid):
    """Fast replacement for groupby('q').t.apply(','.join); keeps within-group order."""
    df = df.sort_values("q", kind="stable")
    q = df.q.values; t = t_eid[df.t.values]
    uq, start = np.unique(q, return_index=True)
    return dict(zip(uq.tolist(), (",".join(x) for x in np.split(t, start[1:]))))

starts = list(range(0, len(Q_te), S1_CHUNK))
print(f"TEST inference: {len(starts)} chunks of {S1_CHUNK:,} S1 | candidates: "
      f"{'GPU exact kNN' if GPU_KNN else 'FAISS CPU'} + rare/address keys on {N_PROC} CPU processes | "
      f"features: {N_PROC} CPU processes | model: {'XGBoost GPU' if MODEL == 'xgb' else 'LightGBM CPU'}")
for ci, a in enumerate(starts):
    kp = os.path.join(CKPT, f"kept_{ci:04d}.pkl")
    cp = os.path.join(CKPT, f"cand_{ci:04d}.tsv")
    if os.path.exists(kp) and os.path.exists(cp):
        continue  # chunk already done

    t0 = time.time()
    qdf = Q_te.iloc[a:a + S1_CHUNK]
    c = BLK_te.candidates(qdf); c["q"] += a
    lists = join_lists(c, t_eid)
    with open(cp + ".tmp", "w", encoding="utf-8", newline="") as fc:
        fc.writelines(S1_TEST_IDS[qi] + "\t" + lists.get(qi, "") + "\n" for qi in range(a, a + len(qdf)))
    t1 = time.time()

    F = make_features_fast(c, Q_te, T_te, FS_te)
    _miss = [col for col in FEATS if col not in F]
    if _miss:   # e.g. the reverse kNN failed on TEST but worked on train: XGBoost / LightGBM treat NaN as missing
        print("  WARNING: features missing at test time, filled with NaN:", _miss)
        for col in _miss: F[col] = np.nan
    t2 = time.time()

    X = F[FEATS]
    p = np.mean([m.predict(X, num_iteration=m.best_iteration, num_threads=N_THREADS) for m in models],
                axis=0).astype(np.float32)
    keep = p >= KEEP_P
    K_chunk = pd.DataFrame({"q": F.q.values[keep], "t": F.t.values[keep], "p": p[keep],
                            "unseen": unseen_q[F.q.values[keep]]})
    for col in (S2_BASE if USE_STAGE2 else []): K_chunk[col] = F[col].values[keep]
    K_chunk.to_pickle(kp + ".tmp"); del K_chunk
    t3 = time.time()

    os.replace(cp + ".tmp", cp); os.replace(kp + ".tmp", kp)   # mark chunk complete
    ncand = len(c)
    del c, F, X, p, lists; gc.collect()
    tic(f"  S1 {min(a + S1_CHUNK, len(Q_te)):,}/{len(Q_te):,} | cands/S1 {ncand/len(qdf):.0f} | "
        f"cand {t1-t0:.0f}s  feat {t2-t1:.0f}s  pred {t3-t2:.0f}s | {gpu_mem()}"
        + (f" | predict: {getattr(XGBModel, 'path', '?')}" if ci == 0 and MODEL == "xgb" else ""))

# ---- merge chunks ----
cand_path = os.path.join(RUN_DIR, "candidate_pairs.tsv")
with open(cand_path, "w", encoding="utf-8", newline="") as fc:
    fc.write("source1_entity_id\tcandidate_entity_ids\n")
    for ci in range(len(starts)):
        with open(os.path.join(CKPT, f"cand_{ci:04d}.tsv"), encoding="utf-8") as f:
            shutil.copyfileobj(f, fc)

K_te = pd.concat([pd.read_pickle(os.path.join(CKPT, f"kept_{ci:04d}.pkl")) for ci in range(len(starts))],
                 ignore_index=True)

def final_decide(K):
    """Apply stage 2 (if it helped on OOF) and per-country thresholds; France uses the global ones + margin."""
    K = K.copy()
    K["is_s3"] = (T_te.src.values[K.t.values] == 3).astype(np.int8)
    if USE_STAGE2:
        K = ctx_feats(K)
        if USE_CE: tic(f"cross-encoder on {len(K):,} kept test pairs"); K = add_ce_feats(K, Q_te, T_te)
        K["p2"] = np.mean([m.predict(K[S2_ALL], num_iteration=m.best_iteration) for m in models2],
                          axis=0).astype(np.float32)
    qc = Q_te.country_key.astype(str).values[K.q.values]
    out = []
    for c in np.unique(qc):
        cfg = CFG_C.get(c, CFG_GLOBAL) if USE_PER_COUNTRY else CFG_GLOBAL
        out.append(decide(K[qc == c].sort_values(P_COL, ascending=False), P_COL,
                          margin=UNSEEN_COUNTRY_MARGIN, **cfg))
    return pd.concat(out, ignore_index=True)

tic("final decision"); sel = final_decide(K_te)
match_lists = join_lists(sel.sort_values(["q", P_COL], ascending=[True, False]), t_eid)

match_path = os.path.join(RUN_DIR, "matching_results.tsv")
with open(match_path, "w", encoding="utf-8", newline="") as f:
    f.write("source1_entity_id\tmatched_entity_ids\n")
    for qi, sid in enumerate(S1_TEST_IDS):
        f.write(sid + "\t" + match_lists.get(qi, "") + "\n")

n_pred = np.bincount(sel.q.values, minlength=len(Q_te))
print(f"predicted singletons: {(n_pred == 0).mean()*100:.2f}%  (train had ~5.6%) | mean matches/S1: {n_pred.mean():.2f}")
print("per country mean matches:", pd.Series(n_pred).groupby(Q_te.country_key.astype(str).values).mean().round(2).to_dict())
if "OOF_STATS" in globals():   # same decision rule on train out-of-fold: a big difference = test is not like train
    _g = pd.Series(n_pred).groupby(Q_te.country_key.astype(str).values)
    _m, _z = _g.mean().to_dict(), (_g.apply(lambda x: (x == 0).mean()) * 100).to_dict()
    print("predicted matches per S1 — test vs train out-of-fold [true train value]:")
    for c in sorted(_m):
        o = OOF_STATS["pred"].get(c)
        print(f"   {c:8s} test {_m[c]:.2f} ({_z[c]:.1f}% none) | " + (
            f"OOF {o:.2f} ({OOF_STATS['none'][c]:.1f}% none) [true {OOF_STATS['true'][c]:.2f}]" if o is not None
            else "not in train"))


## 14. Validate the submission

In [ ]:
def self_check():
    """Streaming check of every submission rule (safe for a ~1 GB candidate file)."""
    ids1 = set(S1_TEST_IDS); valid = set(t_eid.tolist())
    cands = {}
    for fn, col in [(cand_path, "candidate_entity_ids"), (match_path, "matched_entity_ids")]:
        seen = set()
        with open(fn, encoding="utf-8") as f:
            assert f.readline().rstrip("\n") == f"source1_entity_id\t{col}", f"{fn}: header"
            for line in f:
                sid, lst = line.rstrip("\n").split("\t")
                assert sid not in seen, f"{fn}: duplicate row {sid}"; seen.add(sid)
                ids = [x for x in lst.split(",") if x]
                assert len(ids) == len(set(ids)), f"{fn}: duplicate id in {sid}"
                assert all(x in valid for x in ids), f"{fn}: unknown id in {sid}"
                if col == "candidate_entity_ids": cands[sid] = lst
                else: assert set(ids) <= set(cands[sid].split(",")), f"match not in candidates: {sid}"
        assert seen == ids1, f"{fn}: S1 rows missing/extra"
    print("self-check PASS")
self_check()

VALIDATOR = os.path.join(BASE, "utils", "validate_submission.py")
if os.path.exists(VALIDATOR):
    r = subprocess.run([sys.executable, VALIDATOR, "--matching", match_path, "--candidate", cand_path,
                        "--test-dir", TEST_DIR], capture_output=True, text=True, cwd=BASE)
    print(r.stdout[-3000:], r.stderr[-2000:])
else:
    print("official validator not found at", VALIDATOR)

print(f"\nfinal OOF F0.5 of this run: {FINAL_OOF:.4f}  (previous submission: 0.9316)")
save_to_drive(match_path, cand_path)
if IN_COLAB:
    print("UPLOAD THIS FILE →", os.path.join(DRIVE_OUT, "runs", os.path.basename(RUN_DIR), "matching_results.tsv"),
          "(Google Drive)")
else:
    print("UPLOAD THIS FILE →", match_path)
    print("previous submission is untouched →", os.path.join(OUT_DIR, "matching_results.tsv"))

## 15. Summary of the submission file (reads only the files)

In [ ]:
m = pd.read_csv(match_path, sep="\t", dtype=str, keep_default_na=False, quoting=csv.QUOTE_NONE)
s1 = pd.read_csv(os.path.join(TEST_DIR, "test_source1.tsv"), sep="\t", dtype=str, keep_default_na=False,
                 quoting=csv.QUOTE_NONE, usecols=["entity_id", "country"])
print(f"matching_results.tsv → {os.path.getsize(match_path)/1e6:,.1f} MB | "
      f"candidate_pairs.tsv → {os.path.getsize(cand_path)/1e6:,.1f} MB")
print(f"rows: {len(m):,} (should be {len(s1):,}) | unique S1 ids: {m.source1_entity_id.nunique():,}")
m["n"] = m.matched_entity_ids.map(lambda s: len([x for x in s.split(",") if x]))
df = m.merge(s1, left_on="source1_entity_id", right_on="entity_id", how="left")
df["country"] = df.country.str.strip().str.lower()
print(df.groupby("country").agg(S1_records=("n", "size"), mean_matches=("n", "mean"),
                                no_match_pct=("n", lambda x: (x == 0).mean() * 100)).round(2))